# 02 Experiments: graph and tabular representations, classifiers, bootstrap, repeated CV

Reads `phenotypes.tsv` (from `01_preprocessing.ipynb`) and the MLTE embeddings from `PREPROCESSED_DIR`, builds every representation fold by fold in memory, fits the classifiers, and writes:

* participant-level, to `PREPROCESSED_DIR` (never commit): `splits_all_repeats.parquet`, `ckpt_*/` checkpoints, `preds_*.pkl`
* aggregate, to `RESULTS_DIR` (default `results/`): `results_per_fold_*.csv`, `results_summary_*.csv`, `bootstrap_*.csv`, `rcv_*.csv`

Run the cells in order. The run cell and the repeated-CV cell are resumable from their checkpoints.

## Selecting tabular features

In [ ]:
import pandas as pd
import numpy as np
import os

SEED = 12345
np.random.seed(SEED)

from paths import PREPROCESSED_DIR, RESULTS_DIR
preprocessed_dir = PREPROCESSED_DIR   # participant-level inputs / checkpoints / predictions (outside the repository)
os.makedirs(RESULTS_DIR, exist_ok=True)   # aggregate result tables only
# Stamped into every arm folder by apply_split_and_write_to_file and checked before fitting, so arms
# left over from an older build (e.g. different HNSW M / seeding) can never be mixed into a run.
BUILD_TAG = 'exact_nested_v1_mltesplit'   # exact neighbour search + nested tuning; new tag so old checkpoints are never reused
# Coding of the onset-age columns used by every 'onset' arm (raw_onset, *_onset, ablations):
#   'scaled'    : presence indicator + z-scored onset age (train mean/SD among observed; absent -> 0)  [default, 2 x 203 columns]
#   'minus_one' : legacy coding, absent -> -1, present -> raw age in years (203 columns)  [reproduces the earlier results]
ONSET_CODING = 'scaled'
assert ONSET_CODING in ('scaled', 'minus_one')
# Checkpoints are reused when BUILD_TAG and the arm list match, so the tag MUST change with the coding or stale
# 'minus_one' predictions would be silently mixed into a 'scaled' run (and vice versa).
if ONSET_CODING != 'minus_one':
    BUILD_TAG = BUILD_TAG + '_onset' + ONSET_CODING

feature_columns = [
    'age_at_assessment',
    'sex',
    'townsend_deprivation_index',
    'body_fat_percentage_at_assessment',
    'bmi_at_assessment',
    'sleeping_duration_at_assessment',
    'ever_smoked_at_assessment',
    'alcohol_intake_at_assessment__Daily_or_almost_daily',
    'alcohol_intake_at_assessment__Never',
    'alcohol_intake_at_assessment__Once_or_twice_a_week',
    'alcohol_intake_at_assessment__One_to_three_times_a_month',
    'alcohol_intake_at_assessment__Special_occasions_only',
    'alcohol_intake_at_assessment__Three_or_four_times_a_week',
    'able_to_confide_at_assessment__2_4_times_a_week',
    'able_to_confide_at_assessment__About_once_a_month',
    'able_to_confide_at_assessment__About_once_a_week',
    'able_to_confide_at_assessment__Almost_daily',
    'able_to_confide_at_assessment__Do_not_know',
    'able_to_confide_at_assessment__Never_or_almost_never',
    'able_to_confide_at_assessment__Once_every_few_months',
    'leisure_social_activity_at_assessment__Adult_education_class',
    'leisure_social_activity_at_assessment__None_of_the_above',
    'leisure_social_activity_at_assessment__Other_group_activity',
    'leisure_social_activity_at_assessment__Pub_or_social_club',
    'leisure_social_activity_at_assessment__Religious_group',
    'leisure_social_activity_at_assessment__Sports_club_or_gym',
    'genotyping_array',
    'p399_i0_a2',
    'p4282_i0',
    'p20016_i0',
    'p20023_i0',
    'p23444_i0',
    'p23451_i0',
    'p23445_i0',
    'p23459_i0',
    'p23452_i0',
    'father_alzheimer',
    'father_highbloodpressure',
    'father_severedepression',
    'father_parkinson',
    'mother_alzheimer',
    'mother_highbloodpressure',
    'mother_severedepression',
    'mother_parkinson',
    'PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9', 'PC_10',
    'PC_11', 'PC_12', 'PC_13', 'PC_14', 'PC_15', 'PC_16', 'PC_17', 'PC_18', 'PC_19', 'PC_20',
]

icd10_codes_of_positive_class = [
    'F32', # Major depressive disorder, single episode
    # 'G30', # Alzheimer's
    # 'F00', 'F01', 'F02', 'F03', 'G30' # Dementia
]

prediction_horizon = 15  # years

# End of linked outcome data.
# dates for every source you use (use the earliest end date).
data_cutoff = pd.Timestamp('2023-03-31')

df_phenotypes = pd.read_csv(os.path.join(preprocessed_dir, "phenotypes.tsv"), sep="\t")
all_icd10_codes = ['A04', 'A09', 'A41', 'B01', 'B02', 'B05', 'B07', 'B26', 'B34', 'B35', 'B36', 'B37', 'B95', 'B96', 'D50', 'D64', 'D70', 'E03', 'E05', 'E11', 'E14', 'E66', 'E78', 'E86', 'E87', 'F00', 'F01', 'F02', 'F03', 'F10', 'F17', 'F32', 'F41', 'F43', 'G30', 'G40', 'G43', 'G44', 'G45', 'G47', 'G55', 'G56', 'H00', 'H02', 'H04', 'H10', 'H11', 'H25', 'H26', 'H33', 'H35', 'H40', 'H43', 'H52', 'H53', 'H57', 'H60', 'H61', 'H65', 'H66', 'H81', 'H83', 'H90', 'H91', 'H92', 'H93', 'I10', 'I20', 'I21', 'I25', 'I26', 'I44', 'I47', 'I48', 'I49', 'I50', 'I51', 'I63', 'I64', 'I67', 'I73', 'I80', 'I83', 'I84', 'I95', 'J00', 'J01', 'J02', 'J03', 'J06', 'J11', 'J18', 'J20', 'J22', 'J30', 'J31', 'J32', 'J33', 'J34', 'J40', 'J44', 'J45', 'J90', 'J98', 'K04', 'K13', 'K20', 'K21', 'K22', 'K25', 'K26', 'K29', 'K30', 'K31', 'K37', 'K40', 'K42', 'K43', 'K44', 'K51', 'K52', 'K56', 'K57', 'K58', 'K59', 'K60', 'K62', 'K63', 'K64', 'K66', 'K76', 'K80', 'K81', 'K92', 'L02', 'L03', 'L08', 'L20', 'L21', 'L29', 'L30', 'L40', 'L50', 'L57', 'L60', 'L70', 'L71', 'L72', 'L82', 'L90', 'L91', 'L98', 'M06', 'M10', 'M13', 'M15', 'M16', 'M17', 'M19', 'M20', 'M23', 'M25', 'M43', 'M47', 'M48', 'M51', 'M53', 'M54', 'M65', 'M67', 'M70', 'M72', 'M75', 'M76', 'M77', 'M79', 'M81', 'M85', 'N17', 'N18', 'N20', 'N23', 'N28', 'N30', 'N32', 'N39', 'N40', 'N48', 'N50', 'N60', 'N63', 'N64', 'N80', 'N81', 'N83', 'N84', 'N85', 'N87', 'N89', 'N90', 'N92', 'N93', 'N94', 'N95', 'O03', 'O70', 'O80']

age = df_phenotypes["age_at_assessment"]
target = df_phenotypes[icd10_codes_of_positive_class]

# ---------------------------------------------------------------------------
# Cohort selection
# Inclusion depends ONLY on baseline date (identical rule for cases and controls):
# baseline must be at least `prediction_horizon` years before the end of linked data.
# ---------------------------------------------------------------------------
assessment_date = pd.to_datetime(df_phenotypes["date_of_assessment"], errors="coerce")
latest_baseline = data_cutoff - pd.DateOffset(years=prediction_horizon)
followup_selected = assessment_date <= latest_baseline      # missing date -> excluded

# Prevalent target at or before baseline -> excluded (incident prediction)
prevalent_mask = target.le(age, axis=0).any(axis=1)

# Outcome: first target diagnosis in (baseline, baseline + horizon]
incident_mask = (
    target.gt(age, axis=0) & target.le(age + prediction_horizon, axis=0)
).any(axis=1)

# Death within the horizon without the target: kept as a non-event (report this)
early_death_no_target = (
    (df_phenotypes["age_at_death"] < age + prediction_horizon)
    & ~incident_mask & ~prevalent_mask
)

mask = followup_selected & ~prevalent_mask

print(f"Participants before cohort selection:              {len(df_phenotypes)}")
print(f"Baseline on/before {latest_baseline.date()}:               {int(followup_selected.sum())}")
print(f"  of which prevalent {icd10_codes_of_positive_class} (excluded):          {int((followup_selected & prevalent_mask).sum())}")
print(f"  of which died <{prediction_horizon}y without target (kept as non-events): {int((mask & early_death_no_target).sum())}")
print(f"Analysis cohort:                                   {int(mask.sum())}")

Y = incident_mask.loc[mask].reset_index(drop=True).astype(int)
df_phenotypes = df_phenotypes.loc[mask].reset_index(drop=True)
age = df_phenotypes["age_at_assessment"]

# Features use pre-baseline history only: censor ICD-10 events at or after baseline
df_phenotypes[all_icd10_codes] = df_phenotypes[all_icd10_codes].mask(
    df_phenotypes[all_icd10_codes].ge(age, axis=0), np.nan
)
df_phenotypes = df_phenotypes.copy()

X = df_phenotypes[feature_columns].astype('float')

p = Y.mean()
print(f"Incident {icd10_codes_of_positive_class} within {prediction_horizon}y: "
      f"n={len(Y)}, cases={int(Y.sum())}, prevalence={p:.3%}")
print(f"F1 if everyone were predicted positive: {2*p/(1+p):.3f}")

## Splits and per-fold statistics (in memory)

In [ ]:
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

# =====================================================================================
#  IN-MEMORY PIPELINE.  Nothing is written to disk except a tiny per-fold checkpoint of predictions/metrics.
#  Fold-outer: for one fold we build every arm ONE AT A TIME, fit its models, keep only predictions + metrics,
#  and throw the feature matrix away.  Peak memory ~ one arm (<= 0.3 GB) + a few cached blocks (~1 GB).
# =====================================================================================
num_folds = 10
EXPECTED_N_FEATURES = 62

n_total = df_phenotypes.shape[0]
assert len(X) == len(Y) == n_total, (len(X), len(Y), n_total)
assert X.index.equals(pd.RangeIndex(n_total)) and Y.index.equals(X.index), 'X / Y must share a 0..n-1 index'

Y_np = Y.to_numpy().astype(int)

# Pre-baseline presence of each vocabulary category (df_phenotypes was censored at baseline in the cohort cell)
disease_present_all = df_phenotypes[all_icd10_codes].notna().to_numpy()

kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
splits = []
for fold, (trainval_idx, test_idx) in enumerate(kf.split(np.arange(n_total))):
    # validation = 1/9 of the remaining 90%  ->  10% of the cohort (about 54.9k / 6.86k / 6.86k)
    train_idx, val_idx = train_test_split(trainval_idx, test_size=1.0 / 9.0, random_state=SEED + fold)
    assert np.array_equal(np.sort(np.concatenate([train_idx, val_idx, test_idx])), np.arange(n_total))
    splits.append((train_idx, val_idx, test_idx))
train_indices, val_indices, test_indices = (list(t) for t in zip(*splits))
assert np.array_equal(np.sort(np.concatenate(test_indices)), np.arange(n_total))   # the bootstrap relies on this


def make_fold_ctx(fold):
    """Raw-feature block (train-mean imputed, train z-scored) and train-only disease weights for ONE fold."""
    train_idx, val_idx, test_idx = splits[fold]
    X_train = X.iloc[train_idx]
    valid_cols = X_train.columns[X_train.notna().any()]                # drop columns entirely NaN in train
    dropped_cols = [c for c in X.columns if c not in valid_cols]
    X_imputed = X[valid_cols].fillna(X_train[valid_cols].mean())        # mean imputation, TRAIN statistics only
    scaler = StandardScaler().fit(X_imputed.iloc[train_idx])            # normalisation, TRAIN statistics only
    X_norm = np.ascontiguousarray(scaler.transform(X_imputed), dtype=np.float64)
    assert not np.isnan(X_norm).any(), 'NaN left after imputation'
    assert X_norm.shape[1] == EXPECTED_N_FEATURES, (X_norm.shape[1], dropped_cols)

    n_c = disease_present_all[train_idx].sum(axis=0)                    # w_c = min[log((n_train+1)/(n_c+1)), 5], TRAIN only
    w_all = np.minimum(np.log((len(train_idx) + 1) / (n_c + 1)), 5.0)

    is_train = np.zeros(n_total, dtype=bool)
    is_train[train_idx] = True
    print(f'fold {fold}: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)} '
          f'(test events={int(Y_np[test_idx].sum())}); features={X_norm.shape[1]}, dropped={dropped_cols}')
    return dict(fold=fold, repeat=globals().get('CURRENT_REPEAT', 0), train=train_idx, val=val_idx, test=test_idx,
                X_norm=X_norm, w_all=w_all, is_train=is_train)

## Split definitions and export (shared by the main run, repeated CV and external MLTE training)
Single source of truth for every split. `make_rcv_splits(rep)` is deterministic (seeds shifted by 1000 * rep; repeat 0 == main run, asserted). The export writes train / val / test membership keyed by `IID` for repeats 0..`RCV_EXTRA_REPEATS`, plus a `train_hash` per (repeat, fold) that the external MLTE training script must copy into its embedding files so the notebook can refuse embeddings trained on a different training set. Re-run (and retrain MLTE) if the cohort, row order, `SEED` or `num_folds` change.

In [ ]:
import hashlib

RCV_EXTRA_REPEATS = 2        # repeats ADDED to the main run -> 1 + 2 = 3 x 10-fold in total (used by the repeated-CV cell)
SPLITS_EXPORT_PATH = os.path.join(preprocessed_dir, 'splits_all_repeats.parquet')


def make_rcv_splits(rep):
    """Same recipe as the main run (cell 4), with seeds shifted by 1000 * rep (rep = 0 reproduces the main splits)."""
    kf_r = KFold(n_splits=num_folds, shuffle=True, random_state=SEED + 1000 * rep)
    out = []
    for fold, (tv, te) in enumerate(kf_r.split(np.arange(n_total))):
        tr, va = train_test_split(tv, test_size=1.0 / 9.0, random_state=SEED + 1000 * rep + fold)
        assert np.array_equal(np.sort(np.concatenate([tr, va, te])), np.arange(n_total))
        out.append((tr, va, te))
    assert np.array_equal(np.sort(np.concatenate([s[2] for s in out])), np.arange(n_total))
    return out


# repeat 0 must be the main run, otherwise its results cannot be reused
_s0 = make_rcv_splits(0)
assert all(np.array_equal(a, b) for sa, sb in zip(_s0, splits) for a, b in zip(sa, sb)), 'repeat 0 splits differ from the main run'
del _s0

_IID = df_phenotypes['IID'].to_numpy()
assert pd.Series(_IID).is_unique, 'IID must be unique to key the exported splits'


def split_hash(train_idx):
    """Order-independent fingerprint of a training set (IIDs). Same function must be used to stamp / check MLTE embeddings."""
    return hashlib.md5(','.join(sorted(map(str, _IID[np.asarray(train_idx)]))).encode()).hexdigest()


def export_split_indices(path=SPLITS_EXPORT_PATH, n_reps=1 + RCV_EXTRA_REPEATS):
    """One row per (repeat, fold, participant): role in {train, val, test}; train_hash identifies the fold's training set."""
    parts = []
    for rep in range(n_reps):
        for fold, (tr, va, te) in enumerate(make_rcv_splits(rep)):
            h = split_hash(tr)
            for role, idx in (('train', tr), ('val', va), ('test', te)):
                parts.append(pd.DataFrame({'IID': _IID[idx], 'repeat': rep, 'fold': fold, 'role': role, 'train_hash': h}))
    out = pd.concat(parts, ignore_index=True)
    # every participant appears exactly once per (repeat, fold), and exactly once as test per repeat
    assert (out.groupby(['repeat', 'fold'])['IID'].nunique() == n_total).all()
    assert (out[out['role'] == 'test'].groupby('repeat')['IID'].nunique() == n_total).all()
    out.to_parquet(path, index=False)
    return out


_exp = export_split_indices()
print(f'exported {len(_exp):,} rows ({_exp["repeat"].nunique()} repeats x {num_folds} folds x {n_total:,} participants) -> {SPLITS_EXPORT_PATH}')
del _exp


## Graph builders, onset blocks, propagation

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import hnswlib
from scipy import sparse
from joblib import Parallel, delayed
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------------ settings
graph_k = 50                     # neighbours per node (k=50 chosen with logistic regression on validation partitions)
graph_propagation_hops = (1, 10) # hop 0 (the raw features) is always kept
HNSW_M = 64
HNSW_THREADS = -1                # -1 = all cores (graphs are not bit-reproducible with >1 thread)
control_seed = 12345
hnsw_seed = control_seed
oao_large_pool = 500
tau_a = 5.0                      # OAO timing kernel (years)
CHECK_RECALL = True              # HNSW recall vs exact search on 500 sampled rows per graph/fold (~1-2 s each)
REPORT_GRAPHS = True             # cheap per-graph diagnostics (sex/age/code-count assortativity, empty rows)

_T0 = time.time()
def _log(msg):
    print(f'[{time.time() - _T0:8.0f}s] {msg}', flush=True)

# ------------------------------------------------------------------ fold-independent vocabulary tables
code_groups = {
    'dep_adjacent': ['F41', 'F43'],
    'sex_specific': ['N40', 'N48', 'N50', 'N60', 'N63', 'N64', 'N80', 'N81', 'N83', 'N84', 'N85',
                     'N87', 'N89', 'N90', 'N92', 'N93', 'N94', 'N95', 'O03', 'O70', 'O80'],
    'primary_care_type': ['A04', 'A09', 'B01', 'B02', 'B05', 'B07', 'B26', 'B34', 'B35', 'B36', 'B37',
                          'B95', 'B96', 'H00', 'H10', 'H60', 'H65', 'H66', 'J00', 'J01', 'J02', 'J03',
                          'J06', 'J11', 'J20', 'J22', 'L02', 'L03', 'L08'],
    'pain_functional': ['G43', 'G44', 'G47', 'K30', 'K58', 'K59', 'M25', 'M54', 'M79'],
    'other_psych': ['F00', 'F01', 'F02', 'F03', 'F10', 'F17'],
}
code_groups = {g: [c for c in cs if c in all_icd10_codes] for g, cs in code_groups.items()}
ablated_vocabularies = {
    'no_F41_F43': [c for c in all_icd10_codes if c not in code_groups['dep_adjacent']],
    'no_primary_care_type': [c for c in all_icd10_codes if c not in code_groups['primary_care_type']],
}
NOF_TAG = '_noF41F43'
_code_pos = {c: i for i, c in enumerate(all_icd10_codes)}

onset_all = df_phenotypes[all_icd10_codes].astype('float')          # NaN = no pre-baseline diagnosis
onset64 = onset_all.to_numpy(np.float64)
onset32 = onset64.astype(np.float32)
present_all = ~np.isnan(onset32)
baseline_age32 = np.asarray(df_phenotypes['age_at_assessment'], dtype=np.float32)
age64 = df_phenotypes['age_at_assessment'].to_numpy(np.float64)
sex64 = df_phenotypes['sex'].to_numpy(np.float64)
ncodes = present_all.sum(axis=1).astype(np.float64)


def icd10_to_chapter(code):
    return str(code).replace('.', '').upper().strip()[0]


_VOC = {}
def voc(tag=''):
    """Fold-independent tables for the full vocabulary ('') or the vocabulary without F41/F43 (NOF_TAG)."""
    if tag not in _VOC:
        codes = list(all_icd10_codes) if tag == '' else [c for c in all_icd10_codes if c not in code_groups['dep_adjacent']]
        pos = np.array([_code_pos[c] for c in codes])
        on32 = onset32[:, pos]
        rel = on32 - baseline_age32[:, None]
        hist = np.isfinite(rel) & (rel < 0)
        r, c = np.nonzero(hist)
        B = sparse.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=hist.shape)
        B_rec = sparse.csr_matrix((np.exp(rel[r, c] / 5.0).astype(np.float32), (r, c)), shape=hist.shape)  # recency exp(dt/lambda), lambda=5
        _VOC[tag] = dict(codes=codes, pos=pos, onset32=on32, present=~np.isnan(on32), rel=rel, B=B, B_rec=B_rec,
                         groups=np.array([icd10_to_chapter(x) for x in codes]))
    return _VOC[tag]


# ------------------------------------------------------------------ graph operators
def row_normalize(A):
    A = sparse.csr_matrix(A)
    d = np.asarray(A.sum(axis=1)).ravel().astype(np.float64)
    d_inv = np.divide(1.0, d, out=np.zeros_like(d), where=d > 0)
    return sparse.diags(d_inv) @ A


def drop_self(cand, sims, k):
    """Move a row's own index (if retrieved) to the end, keep the first k -> exactly k neighbours per row."""
    is_self = cand == np.arange(cand.shape[0])[:, None]
    order = np.argsort(is_self, axis=1, kind='stable')
    cand = np.take_along_axis(cand, order, axis=1)[:, :k]
    sims = None if sims is None else np.take_along_axis(sims, order, axis=1)[:, :k]
    return cand, sims


def hnsw_knn(V, train_idx, k_query, ef, name, fold, keep_dist=True, batch=50000):
    """HNSW over the TRAINING rows of V, queried with every row in batches (keeps peak memory low at n ~ 370k).
    Returns (participant-index candidates as int32, cosine distances as float32 or None)."""
    V = np.ascontiguousarray(V, dtype=np.float32)
    train_idx = np.asarray(train_idx)
    index = hnswlib.Index(space='cosine', dim=V.shape[1])
    index.init_index(max_elements=len(train_idx), ef_construction=ef, M=HNSW_M, random_seed=hnsw_seed)
    index.set_num_threads(HNSW_THREADS)
    index.add_items(V[train_idx])
    index.set_ef(ef)
    kq = min(k_query, len(train_idx))
    cands, dists = [], []
    for s in range(0, len(V), batch):
        labels, dist = index.knn_query(V[s:s + batch], k=kq)
        cands.append(train_idx[labels.astype(np.int64)].astype(np.int32))
        if keep_dist:
            dists.append(dist)
    del index
    cand = np.vstack(cands)
    if CHECK_RECALL:
        hnsw_recall_check(name, fold, V, train_idx, cand, k_query if k_query <= 50 else kq - 1)
    return cand, (np.vstack(dists) if keep_dist else None)


hnsw_recall_rows = []
def hnsw_recall_check(name, fold, V, train_idx, candidate_indices, k, n_check=500, seed=0):
    """Tie-robust recall@k of the HNSW candidates against exact cosine top-k over the training rows."""
    V = np.asarray(V, dtype=np.float32)
    nrm = np.linalg.norm(V, axis=1)
    Vn = V / np.where(nrm == 0, 1.0, nrm)[:, None]
    train_idx = np.asarray(train_idx)
    pos_in_train = np.full(len(V), -1)
    pos_in_train[train_idx] = np.arange(len(train_idx))
    ok = np.flatnonzero(nrm > 0)
    rows = np.random.default_rng(seed).choice(ok, size=min(n_check, len(ok)), replace=False)
    sims = Vn[rows] @ Vn[train_idx].T
    own = pos_in_train[rows]
    has_self = own >= 0
    sims[np.flatnonzero(has_self), own[has_self]] = -np.inf
    kth = np.partition(sims, -k, axis=1)[:, -k]
    rec = []
    for r, i in enumerate(rows):
        cand = np.asarray(candidate_indices[i])
        cand = cand[cand != i][:k]
        rec.append(np.mean(Vn[cand] @ Vn[i] >= kth[r] - 1e-5))
    hnsw_recall_rows.append({'graph': name, 'fold': fold, 'k': k, 'recall': float(np.mean(rec))})


def propagate(A_norm, Xn, train_idx, hops=graph_propagation_hops):
    """Inductive propagation (training rows are the only sources at every hop) -> [X, X^(1), X^(10)].
    Feature columns are independent, so blocks of columns run in parallel threads."""
    train_idx = np.asarray(train_idx, dtype=int)
    A_ref = sparse.csr_matrix(A_norm)[:, train_idx].tocsr()
    Xv = np.ascontiguousarray(Xn[train_idx])
    keep = set(hops)
    n_threads = max(1, min(os.cpu_count() or 1, 16, Xv.shape[1]))
    blocks = np.array_split(np.arange(Xv.shape[1]), n_threads)

    def run(cols):
        X_ref, res = np.ascontiguousarray(Xv[:, cols]), {}
        for h in range(1, max(hops) + 1):
            X_h = A_ref @ X_ref
            if h in keep:
                res[h] = X_h
            X_ref = np.ascontiguousarray(X_h[train_idx])
        return res

    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(n_threads) as ex:
        parts = list(ex.map(run, blocks))
    return np.hstack([Xn] + [np.hstack([p[h] for p in parts]) for h in sorted(keep)])


graph_report_rows = []
def graph_report(name, fold, A, ctx):
    """What the graph links (sex, age, code count); zero-weight / empty-history rows in test."""
    train_idx, val_idx, test_idx = ctx['train'], ctx['val'], ctx['test']
    A = sparse.csr_matrix(A).copy()
    A.eliminate_zeros()
    ev = np.concatenate([val_idx, test_idx])
    ok = np.asarray(A.sum(axis=1)).ravel() > 0
    rec = {'graph': name, 'fold': fold, 'empty_history_test': int((ncodes[test_idx] == 0).sum()),
           'zero_weight_rows_test': int((~ok[test_idx]).sum())}
    same = np.zeros(len(sex64))
    for v in np.unique(sex64[train_idx]):
        sel = sex64 == v
        same[sel] = (A @ sel.astype(np.float64))[sel]
    rec['same_sex'] = float(same[ev][ok[ev]].mean())
    rec['same_sex_expected'] = float(sum((sex64[train_idx] == v).mean() ** 2 for v in np.unique(sex64[train_idx])))
    for label, z in [('age', age64), ('n_codes', ncodes)]:
        a, b = z[ev][ok[ev]], (A @ z)[ev][ok[ev]]
        rec[f'assort_{label}'] = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else np.nan
    graph_report_rows.append(rec)


# ------------------------------------------------------------------ OAO (exact re-ranking, parallel, sparse-aware)
def _oao_chunk(rows, cand, onset_ages, present, w64, W, is_train, k, tau, want_t, want_u):
    """Exact OAO / weighted-Jaccard re-ranking for a block of participants, touching only the codes the
    participant has. Participants with no history are skipped (all similarities 0 -> all-zero row after normalisation)."""
    out = {'t': ([], [], []), 'u': ([], [], [])}
    for i in rows:
        ci = np.flatnonzero(present[i])
        if ci.size == 0:
            continue
        js = cand[i]
        if is_train[i]:
            js = js[js != i]
        if js.size == 0:
            continue
        sub_p = present[np.ix_(js, ci)]
        w_ci = w64[ci]
        inter = sub_p @ w_ci
        jw = inter / (W[i] + W[js] - inter + 1e-12)
        if want_u:
            o = np.argsort(-jw, kind='stable')[:k]
            out['u'][0].append(np.full(o.size, i)); out['u'][1].append(js[o]); out['u'][2].append(jw[o])
        if want_t:
            diff = np.abs(onset_ages[i, ci][None, :] - onset_ages[np.ix_(js, ci)])
            kern = np.exp(-np.nan_to_num(diff, nan=0.0) / tau)
            timing = (sub_p * w_ci * kern).sum(axis=1) / (inter + 1e-12)
            sim = jw * timing
            o = np.argsort(-sim, kind='stable')[:k]
            out['t'][0].append(np.full(o.size, i)); out['t'][1].append(js[o]); out['t'][2].append(sim[o])
    cat = lambda L, dt: np.concatenate(L) if L else np.empty(0, dtype=dt)
    return {key: (cat(v[0], np.int64), cat(v[1], np.int64), cat(v[2], np.float64)) for key, v in out.items()}


def oao_edges(cand, onset_ages, present, weight_vector, is_train, k, tau=5.0,
              want_timed=True, want_untimed=False, n_jobs=-1, chunk=2000):
    n = cand.shape[0]
    w64 = np.asarray(weight_vector, dtype=np.float64)
    W = present.astype(np.float64) @ w64
    blocks = [range(s, min(s + chunk, n)) for s in range(0, n, chunk)]
    res = Parallel(n_jobs=n_jobs)(delayed(_oao_chunk)(b, cand, onset_ages, present, w64, W, is_train,
                                                     k, tau, want_timed, want_untimed) for b in blocks)
    return {key: tuple(np.concatenate([r[key][j] for r in res]) for j in range(3)) for key in ('t', 'u')}


def build_oao(ctx, n_cand, ef, timed=True, untimed=False, name='oao', tag=''):
    """OAO graph(s). n_cand = candidate pool (k+1 for the main OAO, 500 for the large-pool variants).
    tag='' = full vocabulary; tag=NOF_TAG = vocabulary without F41/F43 (disease weights are still the TRAIN-fold weights
    of the remaining codes, exactly as for the MDBT / DCEG no-F41/F43 graphs)."""
    v = voc(tag)
    w = ctx['w_all'][v['pos']].astype(np.float32)
    retrieval = v['present'].astype(np.float32) * np.sqrt(w)[None, :]
    cand, _ = hnsw_knn(retrieval, ctx['train'], n_cand, ef, name, ctx['fold'], keep_dist=False)
    e = oao_edges(cand, v['onset32'], v['present'], w, ctx['is_train'], graph_k, tau=tau_a,
                  want_timed=timed, want_untimed=untimed)
    n = len(cand)
    mk = lambda t: row_normalize(sparse.csr_matrix((t[2].astype(np.float32), (t[0], t[1])), shape=(n, n)))
    return {'t': mk(e['t']) if timed else None, 'u': mk(e['u']) if untimed else None}


# ------------------------------------------------------------------ MDBT
age_grid = np.array([-20, -15, -10, -5, 0], dtype=np.float32)

def mdbt_descriptor(ctx, tag=''):
    """126-d standardised (train statistics) chapter-level cumulative burden levels + rates."""
    v = voc(tag)
    weights = ctx['w_all'][v['pos']].astype(np.float32)
    rel, groups = v['rel'], v['groups']
    burden_levels = np.concatenate([
        np.stack([
            (((~np.isnan(rel[:, groups == g])) & (rel[:, groups == g] < 0) & (rel[:, groups == g] <= a)).astype(np.float32)
             * weights[groups == g]).sum(axis=1)
            for a in age_grid], axis=1)
        for g in np.unique(groups)], axis=1)
    n_chapters = len(np.unique(groups))
    levels_3d = burden_levels.reshape(len(burden_levels), n_chapters, len(age_grid))
    slopes = np.diff(levels_3d, axis=2) / np.diff(age_grid)[None, None, :]
    burden = np.concatenate([burden_levels, slopes.reshape(len(burden_levels), -1)], axis=1)
    tr = ctx['train']
    burden = (burden - burden[tr].mean(0)) / np.where(burden[tr].std(0) == 0, 1, burden[tr].std(0))
    return burden.astype(np.float32)


def mdbt_graph(ctx, burden, tag=''):
    cand, dist = hnsw_knn(burden, ctx['train'], graph_k + 1, max(100, graph_k + 1), f'mdbt{tag}', ctx['fold'])
    sims = np.maximum(1 - dist, 0)
    ki, kw = drop_self(cand, sims, graph_k)
    n = len(burden)
    return row_normalize(sparse.csr_matrix((kw.ravel(), (np.repeat(np.arange(n), graph_k), ki.ravel())), shape=(n, n)))


# ------------------------------------------------------------------ DCEG
def dceg_embedding(ctx, tag=''):
    """Recency-weighted, L2-normalised participant embedding from the train-only disease co-occurrence SVD."""
    v = voc(tag)
    w32 = ctx['w_all'][v['pos']].astype(np.float32)
    tr = ctx['train']
    m = len(v['codes'])
    B_train = v['B'][tr]
    cooc = (B_train.T @ B_train).astype(np.float32)
    cooc.setdiag(0)
    cooc.eliminate_zeros()
    ds = sparse.diags(w32)
    cooc = (ds @ cooc @ ds).tocsr()
    max_dim = min(64, m - 1, len(tr) - 1)
    assert max_dim >= 2
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        emb = TruncatedSVD(n_components=max_dim, random_state=42).fit_transform(cooc).astype(np.float32)
    wh = v['B_rec'] @ sparse.diags(w32)                    # disease weight x recency
    pe = np.asarray(wh @ emb, dtype=np.float32)
    nrm = np.linalg.norm(pe, axis=1, keepdims=True)
    nrm[nrm == 0] = 1.0
    return pe / nrm


def dceg_vector(ctx, pe):
    tr = ctx['train']
    mu, sd = pe[tr].mean(0), pe[tr].std(0)
    return ((pe - mu) / np.where(sd == 0, 1, sd)).astype(np.float32)


def dceg_graph(ctx, pe, tag=''):
    cand, dist = hnsw_knn(pe, ctx['train'], graph_k + 1, max(100, graph_k + 1), f'dceg{tag}', ctx['fold'])
    sims = np.maximum(1.0 - dist, 0.0).astype(np.float32)
    c, s = drop_self(cand, sims, graph_k)
    n = len(pe)
    return row_normalize(sparse.csr_matrix((s.ravel(), (np.repeat(np.arange(n), graph_k), c.ravel())), shape=(n, n), dtype=np.float32))


# ------------------------------------------------------------------ controls
def build_random(ctx):
    n = n_total
    train_arr = np.asarray(ctx['train'])
    rng = np.random.default_rng(control_seed + ctx['fold'])
    nbrs = np.empty((n, graph_k), dtype=np.int64)
    for i in range(n):
        js = rng.choice(train_arr, size=graph_k + 1, replace=False)
        if ctx['is_train'][i]:
            js = js[js != i]
        nbrs[i] = js[:graph_k]
    return row_normalize(sparse.csr_matrix((np.ones(n * graph_k, dtype=np.float32),
                                            (np.repeat(np.arange(n), graph_k), nbrs.ravel())), shape=(n, n)))


def build_agesex(ctx):
    """k nearest TRAINING participants of the same sex, by age; unit weights."""
    n, tr = n_total, np.asarray(ctx['train'])
    Z = np.column_stack([age64, sex64 * 1e6])                              # sex dominates the distance
    nn = NearestNeighbors(n_neighbors=graph_k + 1, algorithm='kd_tree').fit(Z[tr])
    _, pos = nn.kneighbors(Z)
    nbrs, _ = drop_self(tr[pos], None, graph_k)
    return row_normalize(sparse.csr_matrix((np.ones(n * graph_k, dtype=np.float32),
                                            (np.repeat(np.arange(n), graph_k), nbrs.ravel())), shape=(n, n)))


def build_featknn(ctx):
    Z = ctx['X_norm'].astype(np.float32)
    cand, dist = hnsw_knn(Z, ctx['train'], graph_k + 1, max(100, graph_k + 1), 'featknn', ctx['fold'])
    sims = np.maximum(1.0 - dist, 0.0).astype(np.float32)
    c, s = drop_self(cand, sims, graph_k)
    n = len(Z)
    return row_normalize(sparse.csr_matrix((s.ravel(), (np.repeat(np.arange(n), graph_k), c.ravel())), shape=(n, n), dtype=np.float32))


def shuffle_graph(A, groups, rng, mode='nodes'):
    """'rows': each person gets ANOTHER person's neighbourhood (P @ A); 'nodes': isomorphic relabelling (P @ A @ P.T)."""
    n = A.shape[0]
    perm = np.arange(n)
    for g in groups:
        g = np.asarray(g)
        perm[g] = rng.permutation(g)
    P = sparse.csr_matrix((np.ones(n, dtype=A.dtype), (np.arange(n), perm)), shape=(n, n))
    if mode == 'nodes':
        return (P @ A @ P.T).tocsr()
    A_shuf = (P @ A).tocsr()
    A_shuf = (A_shuf - sparse.diags(A_shuf.diagonal())).tocsr()
    A_shuf.eliminate_zeros()
    return row_normalize(A_shuf)


def build_shuffled(ctx, A_src, mode):
    groups = [ctx['train'], ctx['val'], ctx['test']]
    return shuffle_graph(A_src, groups, np.random.default_rng(control_seed + ctx['fold']), mode=mode)


# ------------------------------------------------------------------ onset-age column blocks
def onset_block(train_idx, kind='onset', codes=None):
    """Onset-age columns. kind='onset' resolves to ONSET_CODING (cell 2). Explicit kinds:
    'minus_one': absent -> -1, present -> age (legacy);  'presence': 0/1;
    'scaled': presence + z-scored age (train statistics among observed; absent -> 0).
    Columns with no observed value in the training fold are dropped (removes F32, F00-F02)."""
    if kind == 'onset':
        kind = ONSET_CODING
    pos = np.arange(len(all_icd10_codes)) if codes is None else np.array([_code_pos[c] for c in codes])
    src = onset64[:, pos]
    valid = np.flatnonzero(~np.isnan(src[train_idx]).all(axis=0))
    obs = src[:, valid]
    if kind == 'minus_one':
        return np.where(np.isnan(obs), -1.0, obs)
    if kind == 'presence':
        return (~np.isnan(obs)).astype(np.float64)
    if kind == 'scaled':
        mu, sd = np.nanmean(obs[train_idx], axis=0), np.nanstd(obs[train_idx], axis=0)
        sd = np.where(sd == 0, 1.0, sd)
        return np.hstack([(~np.isnan(obs)).astype(np.float64), np.nan_to_num((obs - mu) / sd, nan=0.0)])
    raise ValueError(kind)


## Exact neighbour search
Replaces HNSW for every graph (same function name and signature, so the builders above are unchanged). Ties are broken at random.

In [ ]:
# =====================================================================================
#  EXACT NEIGHBOUR SEARCH  (replaces HNSW for every graph: MDBT, DCEG, OAO candidate pools, feature-kNN, MLTE)
#
#  Must run after the "Graph builders, onset blocks, propagation" cell and before any cell that builds a graph.
#  It redefines `hnsw_knn` (same signature, same return values), so every graph builder keeps working unchanged.
#
#  Why: HNSW is approximate (recall 0.76 for MDBT, lowest fold 0.59) and not bit-reproducible with several threads.  At about
#  54.9k training participants and <= 207 dimensions an exact search is a few matrix products per fold.
#
#  Ties: participants without pre-baseline history share one descriptor, and many others share descriptors too, so many cosines
#  are EXACTLY tied.  Cosines are quantised to EXACT_TIE_TOL and ties are broken AT RANDOM (seeded by control_seed, fold and graph
#  name), so tied participants link to a random subset of the tied training participants.  Breaking ties by index would link
#  every tied participant to the same few nodes and turn their propagated features into one constant vector.
#  Self-links are excluded for training rows, so k_query neighbours means k_query OTHER participants (the caller's drop_self
#  therefore keeps the best k).
# =====================================================================================
import zlib
import numpy as np

EXACT_SEARCH = True          # False -> fall back to the approximate HNSW implementation defined in the previous cell
EXACT_BATCH = 256            # query rows per matrix product (~0.4 GB peak at n_train ~ 55k)
EXACT_TIE_TOL = 1e-6         # cosines closer than this count as tied
exact_knn_rows = []          # per graph and fold: boundary-tie share and mean neighbour cosine

_hnsw_knn_approx = hnsw_knn  # keep the HNSW version reachable


def exact_cosine_knn(V, train_idx, k, seed, batch=None, tol=None):
    """Exact top-k cosine neighbours among the TRAINING rows for every row of V (self excluded for training rows).
    Returns (neighbours [n, k] int32 participant indices, best first; cosines [n, k] float32; boundary_tie [n] bool).
    boundary_tie[i]: the k-th and (k+1)-th best cosines of row i differ by <= tol (which tied row is kept is then arbitrary).
    All-zero query rows get cosine 0 to everything (their row of A becomes all zero after clipping, as before)."""
    batch = EXACT_BATCH if batch is None else batch
    tol = EXACT_TIE_TOL if tol is None else tol
    V = np.ascontiguousarray(V, dtype=np.float32)
    nrm = np.linalg.norm(V, axis=1)
    Vn = V / np.where(nrm == 0, 1.0, nrm)[:, None]
    train_idx = np.asarray(train_idx)
    T = np.ascontiguousarray(Vn[train_idx])
    n, n_tr = len(V), len(train_idx)
    assert n_tr > k + 1, 'too few training rows for the requested k'
    pos_in_train = np.full(n, -1, dtype=np.int64)
    pos_in_train[train_idx] = np.arange(n_tr)
    rng = np.random.default_rng(seed)
    nbr = np.empty((n, k), dtype=np.int32)
    sim = np.empty((n, k), dtype=np.float32)
    tied = np.zeros(n, dtype=bool)
    for s in range(0, n, batch):
        e = min(s + batch, n)
        S = (Vn[s:e] @ T.T).astype(np.float64)
        S = np.rint(S / tol) * tol                           # float noise must not decide ties
        S += rng.random(S.shape) * (0.4 * tol)               # random tie-break; jitter < tol never reorders distinct levels
        own = pos_in_train[s:e]
        has = np.flatnonzero(own >= 0)
        S[has, own[has]] = -np.inf                           # no self-links
        top = np.argpartition(S, -(k + 1), axis=1)[:, -(k + 1):]
        o = np.argsort(-np.take_along_axis(S, top, 1), axis=1, kind='stable')
        top = np.take_along_axis(top, o, 1)                  # best first
        true_sim = np.einsum('md,mkd->mk', Vn[s:e], T[top])  # un-quantised cosines of the k + 1 retained rows
        nbr[s:e] = train_idx[top[:, :k]]
        sim[s:e] = true_sim[:, :k]
        tied[s:e] = (true_sim[:, k - 1] - true_sim[:, k]) <= tol
        del S, top, o, true_sim
    return nbr, sim, tied


def hnsw_knn(V, train_idx, k_query, ef, name, fold, keep_dist=True, batch=50000):
    """Drop-in replacement for the HNSW search: (candidates as int32 participant indices, cosine distances or None)."""
    if not EXACT_SEARCH:
        return _hnsw_knn_approx(V, train_idx, k_query, ef, name, fold, keep_dist=keep_dist, batch=batch)
    kq = min(int(k_query), len(train_idx) - 2)
    seed = (int(control_seed) * 1000003 + int(fold) * 101 + zlib.crc32(str(name).encode()) % 10007) % (2 ** 32)
    nbr, sim, tied = exact_cosine_knn(V, train_idx, kq, seed)
    ok = np.linalg.norm(np.asarray(V, dtype=np.float32), axis=1) > 0
    exact_knn_rows.append({'graph': name, 'fold': fold, 'k': kq, 'boundary_tie_share': float(tied[ok].mean()),
                           'mean_cosine': float(sim[ok, :min(kq, 50)].mean())})
    _log(f'exact kNN {name} fold {fold}: k={kq}, boundary-tie share {exact_knn_rows[-1]["boundary_tie_share"]:.3f}')
    return nbr, ((1.0 - sim).astype(np.float32) if keep_dist else None)


CHECK_RECALL = False         # recall against exact search is 1 by construction now


## MLTE embedding graph (`mlte_50`)
Learned patient embeddings from `patient_embeddings.parquet` (key `patient_id` = `IID`, vector column `embedding`), used exactly like the other similarity definitions: train-fold standardisation, exact cosine kNN over the training rows, k = 50, row-normalised, inductive propagation of the 62 features (levels 0, 1, 10).

In [ ]:
import pyarrow.parquet as pq
import pyarrow.compute as pc

# ------------------------------------------------------------------ MLTE settings
MLTE_PATH = os.path.join(preprocessed_dir, 'patient_embeddings.parquet')
MLTE_ID_COL, MLTE_EMB_COL = 'IID', 'embedding'
MLTE_PCA_DIM = None      # e.g. 64: PCA fitted on the TRAIN rows only, before standardisation. None = all dimensions.
CURRENT_REPEAT = 0       # set by run_rcv_unit for the repeated-CV repeats; 0 = main run

_MLTE = {}               # holds ONE (repeat, fold) block at a time (~68.6k x d float32)


def _id_key(s):
    s = pd.Series(s)
    if pd.api.types.is_numeric_dtype(s):
        return s.astype('int64').astype(str)
    return s.astype(str).str.strip()


def mlte_raw(repeat=None, fold=0, train_idx=None):
    """(n_total, d) float32 embeddings aligned to df_phenotypes rows, from the model trained on THIS (repeat, fold).
    If train_idx is given, the stamped train_hash is verified against split_hash(train_idx) and the role=='train'
    IIDs are verified against the training set, so embeddings from a different training set are refused."""
    repeat = CURRENT_REPEAT if repeat is None else repeat
    key = (int(repeat), int(fold))
    if _MLTE.get('key') != key:
        _MLTE.clear()
        flt = (pc.field('repeat') == int(repeat)) & (pc.field('fold') == int(fold))
        tbl = pq.read_table(MLTE_PATH, filters=flt,
                            columns=[MLTE_ID_COL, 'repeat', 'fold', 'role', 'train_hash', 'assessment_date', MLTE_EMB_COL]).to_pandas()
        assert len(tbl), f'no embeddings for repeat {repeat}, fold {fold} in {MLTE_PATH}'
        keys = _id_key(tbl[MLTE_ID_COL])
        assert keys.is_unique, f'repeat {repeat} fold {fold}: {int(keys.duplicated().sum())} duplicated {MLTE_ID_COL}'
        assert tbl['train_hash'].nunique() == 1, f'repeat {repeat} fold {fold}: several train_hash values'
        E = np.vstack(tbl[MLTE_EMB_COL].to_numpy()).astype(np.float32)
        pos = pd.Index(keys).get_indexer(_id_key(df_phenotypes['IID']))
        n_missing = int((pos < 0).sum())
        assert n_missing == 0, f'repeat {repeat} fold {fold}: {n_missing} cohort participants have no embedding'
        E = np.ascontiguousarray(E[pos])
        assert np.isfinite(E).all(), 'non-finite values in the MLTE embeddings'
        # sanity: same assessment dates as the cohort
        d_file = pd.to_datetime(tbl['assessment_date']).to_numpy()[pos]
        d_coh = pd.to_datetime(df_phenotypes['date_of_assessment'], errors='coerce').to_numpy()
        assert np.array_equal(d_file, d_coh), 'assessment_date in the embedding file differs from the cohort'
        role = tbl['role'].to_numpy()[pos]
        _MLTE.update(key=key, E=E, hash=tbl['train_hash'].iloc[0], role=role)
        _log(f'MLTE embeddings repeat {repeat} fold {fold}: {E.shape[0]} rows x {E.shape[1]} dims')
        del tbl
    if train_idx is not None:
        h = split_hash(train_idx)
        assert _MLTE['hash'] == h, (f'MLTE embeddings for repeat {repeat} fold {fold} were trained on a different training set '
                                    f'(file hash {_MLTE["hash"]} != notebook hash {h})')
        assert (_MLTE['role'][np.asarray(train_idx)] == 'train').all(), 'role column disagrees with the notebook training set'
    return _MLTE['E']


def mlte_vector(ctx):
    """Fold-specific embedding: optional train-only PCA, then z-score with TRAIN statistics."""
    E, tr = mlte_raw(ctx.get('repeat', CURRENT_REPEAT), ctx['fold'], train_idx=ctx['train']), ctx['train']
    if MLTE_PCA_DIM:
        from sklearn.decomposition import PCA
        E = PCA(n_components=MLTE_PCA_DIM, random_state=42).fit(E[tr]).transform(E).astype(np.float32)
    mu, sd = E[tr].mean(0), E[tr].std(0)
    return ((E - mu) / np.where(sd == 0, 1, sd)).astype(np.float32)


def mlte_graph(ctx, Z):
    cand, dist = hnsw_knn(Z, ctx['train'], graph_k + 1, max(100, graph_k + 1), 'mlte', ctx['fold'])
    sims = np.maximum(1.0 - dist, 0.0).astype(np.float32)
    c, s = drop_self(cand, sims, graph_k)
    n = len(Z)
    return row_normalize(sparse.csr_matrix((s.ravel(), (np.repeat(np.arange(n), graph_k), c.ravel())), shape=(n, n), dtype=np.float32))

## Build arms one at a time, fit, evaluate (fold-outer, with per-fold checkpoints)
Primary classifier: logistic regression tuned by NESTED inner cross-validation on the training partition (validation data are used only for recalibration). Secondary: default L2 logistic regression, default CatBoost, tuned CatBoost (`RUN_TUNED_CATBOOST_ONLY`, separate pass).

In [ ]:
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, brier_score_loss)
import pickle
from sklearn.metrics import log_loss
from experiments_helpers import (calibration_intercept_slope, expected_calibration_error,
                                 logistic_recalibrate, expected_calibration_error_quantile)

# ------------------------------------------------------------------ what to run
# PRIMARY CLASSIFIER = 'LogisticRegressionTuned' (nested-CV-tuned L1/L2).  'LogisticRegression' (default L2, C=1) is now a
# secondary / sensitivity classifier, 'CatBoost' (defaults) and 'CatBoostTuned' (nested-CV-tuned) are the boosted comparators.
# NB: the order inside PLAN lists is deliberately unchanged, because checkpoints are reused only if the plan matches exactly.
ALL_ARMS = [
    # no graph
    'raw', 'raw_onset', 'raw_presence', 'raw_count', 'raw_countgroups', 'raw_onsetscaled', 'raw_onset_m1', 'raw_mdbtvec', 'raw_dcegvec',   # raw_onset_m1 = legacy -1 coding; raw_onsetscaled == raw_onset when ONSET_CODING='scaled'
    'mdbtvec_only', 'dcegvec_only',   # descriptor alone: no raw features, no graph, no propagation
    'raw_onset__no_F41_F43', 'raw_onset__no_primary_care_type',
    # similarity graphs (oao_50 = 51-candidate pool; pool-500 variants are the large-pool OAO)
    'oao_50', 'oaopool500_50', 'wjpool500_50', 'mdbt_50', 'dceg_50', 'mdbt_50_noF41F43', 'dceg_50_noF41F43',
    # graph + extra columns
    'mdbt_50_onset', 'dceg_50_onset', 'oaopool500_50_onset', 'random_50_onset', 'wjpool500_50_presence',
    # F41/F43 removed from OAO and from the combined arms (graph AND onset columns built without F41/F43)
    'oaopool500_50_noF41F43', 'oaopool500_50_noF41F43_onset', 'mdbt_50_noF41F43_onset', 'dceg_50_noF41F43_onset',
    'mdbt_50_mdbtvec', 'dceg_50_dcegvec', 'random_50_mdbtvec', 'random_50_dcegvec',
    # controls
    'random_50', 'agesex_50', 'featknn_50', 'shuf_dceg_50', 'shufrow_dceg_50',
    # MLTE learned embedding: graph, graph + onset columns, and the direct-covariate comparator (raw + embedding columns).
    'mlte_50', 'mlte_50_onset', 'raw_mltevec',
]
# Only the F41/F43-removal arms added after the main run (mdbt_50_noF41F43 / dceg_50_noF41F43 graph-only arms already exist).
NOF_ARMS = ['oaopool500_50_noF41F43', 'oaopool500_50_noF41F43_onset', 'mdbt_50_noF41F43_onset', 'dceg_50_noF41F43_onset']
# To add ONLY these arms to a finished run without recomputing it (a checkpoint is reused only if its arm list matches exactly),
# set before this cell runs:   BUILD_TAG = BUILD_TAG + '_nof';  ONLY_ARMS = NOF_ARMS
# then merge the two preds_*.pkl 'preds' dicts and concatenate the two results_per_fold_*.csv files.

# ------------------------------------------------------------------ NESTED tuning of the two tuned classifiers
# Hyper-parameters are chosen by INNER cross-validation inside the TRAINING partition only; the chosen setting is then refitted on the
# whole training partition.  The validation partition is no longer used for tuning or early stopping - it is used only for the
# logistic recalibration that follows (and for the reporting-only attribute val_logloss_).  Nothing is tuned on test data.
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed

INNER_FOLDS_LR = 5        # inner folds, tuned logistic regression (9 settings x 5 folds = 45 cheap fits per outer fold and arm)
INNER_FOLDS_CB = 3        # inner folds, tuned CatBoost (8 candidates x 3 folds + 1 refit; use 5 if time allows)
INNER_SEED = 12345
INNER_N_JOBS = -1         # joblib workers for the logistic-regression inner loop (CatBoost already uses all threads)

TUNED_GRID = [('l1', C) for C in (0.003, 0.01, 0.03, 0.1, 0.3)] + [('l2', C) for C in (0.001, 0.01, 0.1, 1.0)]


def _make_lr(pen, C, max_iter):
    # L1 (sparse) and L2 logistic regression; liblinear + intercept_scaling=100 keeps the intercept essentially unpenalised
    if pen == 'l1':
        return LogisticRegression(penalty='l1', C=C, solver='liblinear', intercept_scaling=100.0, max_iter=max_iter, tol=1e-3,
                                  random_state=INNER_SEED)   # seeded: unseeded liblinear gives slightly different L1 fits on identical data
    return LogisticRegression(penalty='l2', C=C, solver='lbfgs', max_iter=max_iter)


def _inner_lr_loss(pen, C, max_iter, X, y, tr, te):
    m = _make_lr(pen, C, max_iter)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        m.fit(X[tr], y[tr])
    return log_loss(y[te], m.predict_proba(X[te])[:, 1], labels=[0, 1])


class TunedLogReg:
    """Nested-CV tuned L1/L2 logistic regression (name kept so that MODELS / PLAN / the bootstrap cell work unchanged)."""
    def __init__(self, grid=None, inner_folds=None, max_iter=5000, seed=None, n_jobs=None):
        self.grid = list(grid or TUNED_GRID)
        self.inner_folds = INNER_FOLDS_LR if inner_folds is None else inner_folds
        self.max_iter = max_iter
        self.seed = INNER_SEED if seed is None else seed
        self.n_jobs = INNER_N_JOBS if n_jobs is None else n_jobs

    def fit(self, X, y, X_val=None, y_val=None):
        X, y = np.asarray(X), np.asarray(y)
        splits = list(StratifiedKFold(self.inner_folds, shuffle=True, random_state=self.seed).split(X, y))
        jobs = [(j, f) for j in range(len(self.grid)) for f in range(len(splits))]
        losses = Parallel(n_jobs=self.n_jobs)(
            delayed(_inner_lr_loss)(self.grid[j][0], self.grid[j][1], self.max_iter, X, y, *splits[f]) for j, f in jobs)
        L = np.zeros((len(self.grid), len(splits)))
        for (j, f), l in zip(jobs, losses):
            L[j, f] = l
        self.inner_logloss_ = L.mean(axis=1)                          # mean inner-fold log loss per grid point
        jb = int(np.argmin(self.inner_logloss_))
        self.best_penalty_, self.best_C_ = self.grid[jb]
        self.model_ = _make_lr(self.best_penalty_, self.best_C_, self.max_iter)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            self.model_.fit(X, y)                                     # refit on the whole training partition
        self.n_nonzero_ = int(np.count_nonzero(self.model_.coef_))
        # reporting only: never used to choose anything
        self.val_logloss_ = (log_loss(y_val, self.model_.predict_proba(X_val)[:, 1], labels=[0, 1])
                             if X_val is not None and y_val is not None else np.nan)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


CB_GRID = [dict(depth=d, learning_rate=lr, l2_leaf_reg=l2) for d in (4, 6) for lr in (0.03, 0.1) for l2 in (3.0, 10.0)]   # 8 candidates


class TunedCatBoost:
    """Nested-CV tuned CatBoost.  Each candidate is fitted on the inner-training folds with early stopping (log loss) on the inner
    held-out fold; the candidate with the lowest mean inner log loss wins, and is refitted on the WHOLE training partition with
    the mean of its inner best iteration counts (no early stopping, no validation data)."""
    def __init__(self, grid=None, inner_folds=None, max_iter=2000, patience=100, seed=0, inner_seed=None):
        self.grid = list(grid or CB_GRID)
        self.inner_folds = INNER_FOLDS_CB if inner_folds is None else inner_folds
        self.max_iter, self.patience, self.seed = max_iter, patience, seed
        self.inner_seed = INNER_SEED if inner_seed is None else inner_seed

    def fit(self, X, y, X_val=None, y_val=None):
        X, y = np.asarray(X), np.asarray(y)
        splits = list(StratifiedKFold(self.inner_folds, shuffle=True, random_state=self.inner_seed).split(X, y))
        best = None
        self.inner_logloss_ = []
        for params in self.grid:
            ll, its = [], []
            for tr, te in splits:
                m = CatBoostClassifier(loss_function='Logloss', eval_metric='Logloss', iterations=self.max_iter,
                                       early_stopping_rounds=self.patience, verbose=False, thread_count=-1,
                                       random_seed=self.seed, **params)
                m.fit(X[tr], y[tr], eval_set=(X[te], y[te]), use_best_model=True)
                ll.append(log_loss(y[te], m.predict_proba(X[te])[:, 1], labels=[0, 1]))
                its.append(int(m.get_best_iteration()) + 1)           # number of trees kept
            self.inner_logloss_.append(float(np.mean(ll)))
            if best is None or np.mean(ll) < best[0]:
                best = (float(np.mean(ll)), params, int(max(10, round(float(np.mean(its))))))
        _, self.best_params_, n_iter = best
        self.best_iteration_ = n_iter
        self.model_ = CatBoostClassifier(loss_function='Logloss', iterations=n_iter, verbose=False, thread_count=-1,
                                         random_seed=self.seed, **self.best_params_)
        self.model_.fit(X, y)                                         # whole training partition, fixed number of trees
        self.val_logloss_ = (log_loss(y_val, self.model_.predict_proba(X_val)[:, 1], labels=[0, 1])
                             if X_val is not None and y_val is not None else np.nan)   # reporting only
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

# UNWEIGHTED is the main experiment (the class-weighted variants were removed). Names are kept so that the bootstrap cell
# (CLF = 'LogisticRegression' primary, 'CatBoost' secondary) works unchanged.
MODELS = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=10000),                      # secondary: default L2 (C=1)
    'CatBoost':           lambda: CatBoostClassifier(verbose=False, eval_metric='PRAUC', thread_count=-1, random_seed=0),   # secondary: default CatBoost
    'LogisticRegressionTuned': lambda: TunedLogReg(),                                         # PRIMARY: L1 or L2, C chosen by NESTED inner CV on the training partition
    'CatBoostTuned':      lambda: TunedCatBoost(),                                            # tuned boosted model, NESTED inner CV (see CB_GRID)
}
TCB_ARMS = {'raw','raw_onset','oaopool500_50','mdbt_50','dceg_50','random_50','agesex_50','shufrow_dceg_50',
            'oaopool500_50_onset','mdbt_50_onset','dceg_50_onset','mlte_50','mlte_50_onset','raw_mltevec'}
PLAN = {a: ['LogisticRegressionTuned','LogisticRegression','CatBoost'] + (['CatBoostTuned'] if a in TCB_ARMS else []) for a in ALL_ARMS}
# ---- trim here to save time, e.g. only the primary classifier on some arms:
# PLAN = {a: ['LogisticRegression'] for a in ALL_ARMS}
# PLAN['raw'] += ['CatBoost']   # CatBoost dominates the run time
ONLY_ARMS = None            # e.g. ['raw', 'raw_onset', 'mdbt_50'] for a quick test; None = every arm in PLAN

# To add ONLY the MLTE arms to an existing run without recomputing or overwriting it (checkpoints, preds_*.pkl and
# results_*.csv are all named after BUILD_TAG, and a checkpoint is reused only if its arm list matches exactly), run with:
#   BUILD_TAG = BUILD_TAG + '_mlte'
#   ONLY_ARMS = ['mlte_50', 'mlte_50_onset', 'raw_mltevec']
# set BEFORE this cell runs; then concatenate the two results_per_fold_*.csv files and merge the two preds_*.pkl 'preds' dicts.
FOLDS = range(num_folds)    # e.g. range(3) for a quick test

RESUME = True
CKPT_DIR = f'{preprocessed_dir}/ckpt_{BUILD_TAG}'      # tiny per-fold files (predictions + metrics), not feature matrices
os.makedirs(CKPT_DIR, exist_ok=True)


# ------------------------------------------------------------------ arms: built one at a time
def iter_arms(ctx, wanted):
    """Yield (arm name, feature matrix) one at a time; the caller fits its models and drops the matrix."""
    Xn, tr = ctx['X_norm'], ctx['train']
    need = lambda *names: any(n in wanted for n in names)
    cache = {}
    def blk(kind):
        if kind not in cache:
            cache[kind] = onset_block(tr, kind)
        return cache[kind]

    # ---- arms without a graph
    if need('raw'):
        yield 'raw', Xn
    if need('raw_onset'):
        yield 'raw_onset', np.hstack([Xn, blk('onset')])
    if need('raw_presence'):
        yield 'raw_presence', np.hstack([Xn, blk('presence')])
    if need('raw_onsetscaled'):
        yield 'raw_onsetscaled', np.hstack([Xn, onset_block(tr, 'scaled')])
    if need('raw_onset_m1'):
        yield 'raw_onset_m1', np.hstack([Xn, onset_block(tr, 'minus_one')])
    if need('raw_count', 'raw_countgroups'):
        counts = ncodes[:, None]
        if need('raw_count'):
            yield 'raw_count', np.hstack([Xn, counts])
        if need('raw_countgroups'):
            grp = [present_all[:, [_code_pos[c] for c in cs]].sum(axis=1).astype(np.float64)[:, None] for cs in code_groups.values()]
            yield 'raw_countgroups', np.hstack([Xn, counts] + grp)
    for vname, codes in ablated_vocabularies.items():
        if need(f'raw_onset__{vname}'):
            yield f'raw_onset__{vname}', np.hstack([Xn, onset_block(tr, 'onset', codes)])

    # ---- MDBT (its descriptor also feeds the *_mdbtvec arms)
    mdbt_vec = None
    if need('mdbt_50', 'mdbt_50_onset', 'mdbt_50_mdbtvec', 'raw_mdbtvec', 'random_50_mdbtvec', 'mdbtvec_only'):
        mdbt_vec = mdbt_descriptor(ctx)
        if need('raw_mdbtvec'):
            yield 'raw_mdbtvec', np.hstack([Xn, mdbt_vec])
        if need('mdbtvec_only'):
            yield 'mdbtvec_only', mdbt_vec
    if need('mdbt_50', 'mdbt_50_onset', 'mdbt_50_mdbtvec'):
        A = mdbt_graph(ctx, mdbt_vec)
        if REPORT_GRAPHS: graph_report('mdbt_50', ctx['fold'], A, ctx)
        Xf = propagate(A, Xn, tr); del A
        if need('mdbt_50'):         yield 'mdbt_50', Xf
        if need('mdbt_50_onset'):   yield 'mdbt_50_onset', np.hstack([Xf, blk('onset')])
        if need('mdbt_50_mdbtvec'): yield 'mdbt_50_mdbtvec', np.hstack([Xf, mdbt_vec])
        del Xf

    # ---- DCEG (its embedding feeds *_dcegvec; its graph is the source of the shuffled-history controls)
    dceg_vec, A_dceg = None, None
    if need('dceg_50', 'dceg_50_onset', 'dceg_50_dcegvec', 'raw_dcegvec', 'dcegvec_only', 'random_50_dcegvec', 'shuf_dceg_50', 'shufrow_dceg_50'):
        pe = dceg_embedding(ctx)
        dceg_vec = dceg_vector(ctx, pe)
        if need('raw_dcegvec'):
            yield 'raw_dcegvec', np.hstack([Xn, dceg_vec])
        if need('dcegvec_only'):
            yield 'dcegvec_only', dceg_vec
        if need('dceg_50', 'dceg_50_onset', 'dceg_50_dcegvec', 'shuf_dceg_50', 'shufrow_dceg_50'):
            A_dceg = dceg_graph(ctx, pe)
            if REPORT_GRAPHS: graph_report('dceg_50', ctx['fold'], A_dceg, ctx)
        del pe
    if need('dceg_50', 'dceg_50_onset', 'dceg_50_dcegvec'):
        Xf = propagate(A_dceg, Xn, tr)
        if need('dceg_50'):         yield 'dceg_50', Xf
        if need('dceg_50_onset'):   yield 'dceg_50_onset', np.hstack([Xf, blk('onset')])
        if need('dceg_50_dcegvec'): yield 'dceg_50_dcegvec', np.hstack([Xf, dceg_vec])
        del Xf

    # ---- OAO: 51-candidate pool, and the 500-candidate pool (timed + untimed in one pass)
    if need('oao_50'):
        G = build_oao(ctx, graph_k + 1, max(100, graph_k + 1), timed=True, untimed=False, name='oao')
        if REPORT_GRAPHS: graph_report('oao_50', ctx['fold'], G['t'], ctx)
        Xf = propagate(G['t'], Xn, tr); del G
        yield 'oao_50', Xf
        del Xf
    if need('oaopool500_50', 'oaopool500_50_onset', 'wjpool500_50', 'wjpool500_50_presence'):
        want_t = need('oaopool500_50', 'oaopool500_50_onset')
        want_u = need('wjpool500_50', 'wjpool500_50_presence')
        G = build_oao(ctx, oao_large_pool, max(100, oao_large_pool + 1), timed=want_t, untimed=want_u, name=f'oaopool{oao_large_pool}')
        if want_t:
            if REPORT_GRAPHS: graph_report('oaopool500_50', ctx['fold'], G['t'], ctx)
            Xf = propagate(G['t'], Xn, tr)
            if need('oaopool500_50'):       yield 'oaopool500_50', Xf
            if need('oaopool500_50_onset'): yield 'oaopool500_50_onset', np.hstack([Xf, blk('onset')])
            del Xf
        if want_u:
            if REPORT_GRAPHS: graph_report('wjpool500_50', ctx['fold'], G['u'], ctx)
            Xf = propagate(G['u'], Xn, tr)
            if need('wjpool500_50'):          yield 'wjpool500_50', Xf
            if need('wjpool500_50_presence'): yield 'wjpool500_50_presence', np.hstack([Xf, blk('presence')])
            del Xf
        del G

    # ---- graphs built without F41/F43: graph-only arms, and the combined arms (graph + onset columns, BOTH without F41/F43)
    onset_nof = lambda: onset_block(tr, 'onset', ablated_vocabularies['no_F41_F43'])
    if need('mdbt_50_noF41F43', 'mdbt_50_noF41F43_onset'):
        A = mdbt_graph(ctx, mdbt_descriptor(ctx, NOF_TAG), NOF_TAG)
        Xf = propagate(A, Xn, tr); del A
        if need('mdbt_50_noF41F43'):       yield 'mdbt_50_noF41F43', Xf
        if need('mdbt_50_noF41F43_onset'): yield 'mdbt_50_noF41F43_onset', np.hstack([Xf, onset_nof()])
        del Xf
    if need('dceg_50_noF41F43', 'dceg_50_noF41F43_onset'):
        A = dceg_graph(ctx, dceg_embedding(ctx, NOF_TAG), NOF_TAG)
        Xf = propagate(A, Xn, tr); del A
        if need('dceg_50_noF41F43'):       yield 'dceg_50_noF41F43', Xf
        if need('dceg_50_noF41F43_onset'): yield 'dceg_50_noF41F43_onset', np.hstack([Xf, onset_nof()])
        del Xf
    if need('oaopool500_50_noF41F43', 'oaopool500_50_noF41F43_onset'):
        G = build_oao(ctx, oao_large_pool, max(100, oao_large_pool + 1), timed=True, untimed=False,
                      name=f'oaopool{oao_large_pool}{NOF_TAG}', tag=NOF_TAG)
        if REPORT_GRAPHS: graph_report('oaopool500_50_noF41F43', ctx['fold'], G['t'], ctx)
        Xf = propagate(G['t'], Xn, tr); del G
        if need('oaopool500_50_noF41F43'):       yield 'oaopool500_50_noF41F43', Xf
        if need('oaopool500_50_noF41F43_onset'): yield 'oaopool500_50_noF41F43_onset', np.hstack([Xf, onset_nof()])
        del Xf

    # ---- MLTE (learned embedding). raw_mltevec = raw features + embedding as ordinary columns (the comparator for the graph)
    if need('mlte_50', 'mlte_50_onset', 'raw_mltevec'):
        mlte_vec = mlte_vector(ctx)
        if need('raw_mltevec'):
            yield 'raw_mltevec', np.hstack([Xn, mlte_vec])
        if need('mlte_50', 'mlte_50_onset'):
            A = mlte_graph(ctx, mlte_vec)
            if REPORT_GRAPHS: graph_report('mlte_50', ctx['fold'], A, ctx)
            Xf = propagate(A, Xn, tr); del A
            if need('mlte_50'):       yield 'mlte_50', Xf
            if need('mlte_50_onset'): yield 'mlte_50_onset', np.hstack([Xf, blk('onset')])
            del Xf
        del mlte_vec

    # ---- controls
    if need('random_50', 'random_50_onset', 'random_50_mdbtvec', 'random_50_dcegvec'):
        A = build_random(ctx)
        if REPORT_GRAPHS: graph_report('random_50', ctx['fold'], A, ctx)
        Xf = propagate(A, Xn, tr); del A
        if need('random_50'):         yield 'random_50', Xf
        if need('random_50_onset'):   yield 'random_50_onset', np.hstack([Xf, blk('onset')])
        if need('random_50_mdbtvec'): yield 'random_50_mdbtvec', np.hstack([Xf, mdbt_vec])
        if need('random_50_dcegvec'): yield 'random_50_dcegvec', np.hstack([Xf, dceg_vec])
        del Xf
    if need('agesex_50'):
        A = build_agesex(ctx)
        if REPORT_GRAPHS: graph_report('agesex_50', ctx['fold'], A, ctx)
        Xf = propagate(A, Xn, tr); del A
        yield 'agesex_50', Xf
        del Xf
    if need('featknn_50'):
        A = build_featknn(ctx)
        if REPORT_GRAPHS: graph_report('featknn_50', ctx['fold'], A, ctx)
        Xf = propagate(A, Xn, tr); del A
        yield 'featknn_50', Xf
        del Xf
    for arm, mode in [('shuf_dceg_50', 'nodes'), ('shufrow_dceg_50', 'rows')]:
        if need(arm):
            A = build_shuffled(ctx, A_dceg, mode)
            Xf = propagate(A, Xn, tr); del A
            yield arm, Xf
            del Xf


# ------------------------------------------------------------------ fit + evaluate one arm
def evaluate_arm(arm, X, ctx, results, fold_preds):
    tr, va, te = ctx['train'], ctx['val'], ctx['test']
    has_nan = bool(np.isnan(X).any())                  # normally False: then LR and CatBoost see identical inputs, no extra copies
    Xtr, Xva, Xte = X[tr], X[va], X[te]
    ytr, yva, yte = Y_np[tr], Y_np[va], Y_np[te]
    if has_nan:
        Xtr0, Xva0, Xte0 = (np.where(np.isnan(M), 0.0, M) for M in (Xtr, Xva, Xte))    # NaN -> 0 for logistic regression
    else:
        Xtr0, Xva0, Xte0 = Xtr, Xva, Xte
    for mname in PLAN[arm]:
        t0 = time.time()
        model = MODELS[mname]()
        if mname == 'CatBoostTuned':
            A_va, A_te = Xva, Xte
            model.fit(Xtr, ytr, Xva, yva)                  # inner CV on the training partition chooses the hyperparameters (validation data are used only for recalibration)
        elif mname.startswith('CatBoost'):
            A_va, A_te = Xva, Xte
            model.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
        elif mname.startswith('LogisticRegressionTuned'):
            A_va, A_te = Xva0, Xte0
            model.fit(Xtr0, ytr, Xva0, yva)               # inner CV on the training partition selects penalty + C (validation data are used only for recalibration)
        else:
            A_va, A_te = Xva0, Xte0
            model.fit(Xtr0, ytr)
        y_prob = model.predict_proba(A_te)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        p_val = model.predict_proba(A_va)[:, 1]
        y_prob_cal = logistic_recalibrate(p_val, yva, y_prob)          # fitted on validation, applied to test
        ci, cs = calibration_intercept_slope(yte, y_prob)
        cic, csc = calibration_intercept_slope(yte, y_prob_cal)
        results.append({
            'arm': arm, 'model': mname, 'fold': ctx['fold'], 'n_features': X.shape[1],
            'accuracy': accuracy_score(yte, y_pred),
            'precision': precision_score(yte, y_pred, zero_division=0),
            'recall': recall_score(yte, y_pred, zero_division=0),
            'f1': f1_score(yte, y_pred, zero_division=0),
            'auc': roc_auc_score(yte, y_prob), 'aupr': average_precision_score(yte, y_prob),
            'brier': brier_score_loss(yte, y_prob), 'calibration_intercept': ci, 'calibration_slope': cs,
            'ece': expected_calibration_error(yte, y_prob),
            'brier_cal': brier_score_loss(yte, y_prob_cal), 'calibration_intercept_cal': cic,
            'calibration_slope_cal': csc, 'ece_cal': expected_calibration_error_quantile(yte, y_prob_cal),
            'tuned_penalty': getattr(model, 'best_penalty_', None), 'tuned_C': getattr(model, 'best_C_', np.nan),
            'tuned_n_nonzero': getattr(model, 'n_nonzero_', np.nan),
            'tuned_cb_params': str(getattr(model, 'best_params_', None)), 'tuned_cb_best_iter': getattr(model, 'best_iteration_', np.nan),
            'time': time.time() - t0})
        fold_preds[(arm, mname)] = pd.DataFrame({'row_id': te, 'fold': ctx['fold'], 'y': yte, 'p': y_prob})
        r_ = results[-1]
        _log(f"fold {ctx['fold']} {arm:34s} {mname:30s} p={X.shape[1]:4d} AUC={r_['auc']:.4f} AUPR={r_['aupr']:.4f} ({r_['time']:.0f}s)")


# ------------------------------------------------------------------ fold-outer driver
arms_to_run = [a for a in ALL_ARMS if (ONLY_ARMS is None or a in ONLY_ARMS)]
wanted = set(arms_to_run)
all_results, all_fold_preds = [], {}
graph_report_rows.clear(); hnsw_recall_rows.clear(); exact_knn_rows.clear()
if wanted & {'mlte_50', 'mlte_50_onset', 'raw_mltevec'}:
    # fail fast: validity gate, hash, ID alignment, NaNs for every repeat and fold that will be used
    for _r in range(1 + RCV_EXTRA_REPEATS):
        _sp = make_rcv_splits(_r)
        for _f in range(num_folds):
            mlte_raw(_r, _f, train_idx=_sp[_f][0])

for fold in FOLDS:
    ck = f'{CKPT_DIR}/fold{fold}.pkl'
    if RESUME and os.path.exists(ck):
        d = pickle.load(open(ck, 'rb'))
        if d.get('build') == BUILD_TAG and d.get('arms') == arms_to_run and d.get('plan') == {a: PLAN[a] for a in arms_to_run}:
            all_results += d['results']; graph_report_rows += d['graph_report']; hnsw_recall_rows += d['recall']; exact_knn_rows += d.get('exact', [])
            for k, v in d['preds'].items():
                all_fold_preds.setdefault(k, []).append(v)
            _log(f'fold {fold}: loaded checkpoint, skipped')
            continue
    ctx = make_fold_ctx(fold)
    n_res, n_rep, n_rec, n_ex = len(all_results), len(graph_report_rows), len(hnsw_recall_rows), len(exact_knn_rows)
    fold_preds = {}
    for arm, X_arm in iter_arms(ctx, wanted):          # NB: never call this loop variable X (that is the cohort table)
        assert arm in wanted, arm
        evaluate_arm(arm, X_arm, ctx, all_results, fold_preds)
        del X_arm
    missing = [a for a in arms_to_run if (a, PLAN[a][0]) not in fold_preds]
    assert not missing, f'arms not produced: {missing}'
    for k, v in fold_preds.items():
        all_fold_preds.setdefault(k, []).append(v)
    pickle.dump({'build': BUILD_TAG, 'arms': arms_to_run, 'plan': {a: PLAN[a] for a in arms_to_run},
                 'results': all_results[n_res:], 'graph_report': graph_report_rows[n_rep:],
                 'recall': hnsw_recall_rows[n_rec:], 'exact': exact_knn_rows[n_ex:], 'preds': fold_preds}, open(ck, 'wb'))
    del ctx, fold_preds

# ------------------------------------------------------------------ assemble outputs (same structure as before)
preds = {k: pd.concat(v).sort_values(['fold', 'row_id']).reset_index(drop=True) for k, v in all_fold_preds.items()}
results_df = pd.DataFrame(all_results)
pd.to_pickle({'build': BUILD_TAG, 'preds': preds}, os.path.join(preprocessed_dir, f'preds_{BUILD_TAG}.pkl'))   # participant-level: stays outside the repo
results_df.to_csv(os.path.join(RESULTS_DIR, f'results_per_fold_{BUILD_TAG}.csv'), index=False)

metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'auc', 'aupr', 'brier', 'calibration_intercept', 'calibration_slope',
               'ece', 'brier_cal', 'calibration_intercept_cal', 'calibration_slope_cal', 'ece_cal']
summary = results_df.groupby(['model', 'arm'], sort=False)[metric_cols].agg(['mean', 'std'])
summary.to_csv(os.path.join(RESULTS_DIR, f'results_summary_{BUILD_TAG}.csv'))
print(results_df.groupby(['model', 'arm'], sort=False)[['auc', 'aupr']].agg(['mean', 'std']).round(4).to_string())
if graph_report_rows:
    print(pd.DataFrame(graph_report_rows).drop(columns='fold').groupby('graph', sort=False).mean().round(3).to_string())
if hnsw_recall_rows:
    print('\nHNSW recall@k vs exact search (mean / min over folds):')
    print(pd.DataFrame(hnsw_recall_rows).drop(columns='fold').groupby('graph', sort=False)
          .agg(k=('k', 'first'), recall_mean=('recall', 'mean'), recall_min=('recall', 'min')).round(3).to_string())
if exact_knn_rows:
    print('\nExact kNN diagnostics (mean over folds; boundary_tie_share = k-th and (k+1)-th neighbour tied):')
    print(pd.DataFrame(exact_knn_rows).drop(columns='fold').groupby('graph', sort=False).mean().round(4).to_string())


## Patient-level bootstrap

In [ ]:
# =====================================================================================
#  PATIENT-LEVEL BOOTSTRAP
#
#  Estimand = mean over folds of the per-fold metric (same as the paper's tables).
#  Participants are resampled with replacement WITHIN each test fold; every arm sees the
#  same resampled participants, so differences are paired.  Captures test-set sampling
#  variability only (models are not refitted).
# =====================================================================================
import re
from scipy.stats import norm

# ------------------------- CONFIG (edit only if the checks below complain) -------------------------
B = 2000                # B=2000 for a first pass, 10000 for final numbers (~10-15 min)
# Main run (all arms; tuned LR, default L2 LR, default CatBoost) + the tuned-CatBoost pass (TCB_ARMS only), read from disk so the
# result does not depend on what is currently in the kernel's `preds`.
_MAIN = globals().get('MAIN_BUILD_TAG', BUILD_TAG)
P = pd.read_pickle(os.path.join(preprocessed_dir, f'preds_{_MAIN}.pkl'))['preds']      # {(arm, classifier): table with row_id, fold, y, p}

TEST_IDX = None         # list of K arrays of test-row positions; ONLY needed if the auto-search fails
# role -> arm key in preds. Explicit on purpose: no fuzzy token matching, so oao_50 / oaopool500_50 or
# shuf / shufrow can never be confused. None = arm not run (its contrasts are reported as skipped).
NAME = dict(
    raw='raw', onset='raw_onset',
    oao='oaopool500_50', mdbt='mdbt_50', dceg='dceg_50',              # OAO = pool-500 variant (see run cell comment)
    rand='random_50', agesex='agesex_50', shuf='shufrow_dceg_50',     # shufrow = another person's neighbourhood
    oao_on='oaopool500_50_onset', mdbt_on='mdbt_50_onset', dceg_on='dceg_50_onset',
    presence='raw_presence', code_count='raw_count', count_group='raw_countgroups',
    no_f41='raw_onset__no_F41_F43', no_chF=None, no_pc='raw_onset__no_primary_care_type',
    oao500='oaopool500_50', wj500='wjpool500_50', fknn='featknn_50', shuf_sex=None,
    mlte='mlte_50', mlte_on='mlte_50_onset', mltevec='raw_mltevec',    # MLTE learned-embedding arms (None / absent = skipped)
    # F41/F43 removed (graph and, for *_on_nof, also the onset columns)
    oao_nof='oaopool500_50_noF41F43', mdbt_nof='mdbt_50_noF41F43', dceg_nof='dceg_50_noF41F43',
    oao_on_nof='oaopool500_50_noF41F43_onset', mdbt_on_nof='mdbt_50_noF41F43_onset', dceg_on_nof='dceg_50_noF41F43_onset',
)
STRICT_FAMILIES = False  # True: raise if any of the 31 + 6 pre-specified contrasts cannot be computed
CLASSIFIERS = ['LogisticRegressionTuned', 'LogisticRegression', 'CatBoost', 'CatBoostTuned']   # bootstrapped together on the SAME resamples (PRIMARY = tuned LR, first); keys of preds are (arm, classifier)
USE_ALL = True          # True: bootstrap EVERY arm in preds (all additional arms, both classifiers). False: only the arms used in the paper's contrasts (faster)
# ----------------------------------------------------------------------------------------------------

y = np.asarray(Y).astype(int).ravel()
n = len(y)
IDX_KEYS = ('test_idx', 'idx_test', 'test_index', 'te', 'test', 'idx', 'index', 'indices')
PRB_KEYS = ('y_prob', 'y_proba', 'proba', 'prob', 'y_score', 'score', 'y_pred_proba',
            'test_pred', 'pred', 'y_pred', 'p', 'preds')


# ------------------------------------- 1. find the test folds -------------------------------------
def _find_test_idx():
    """Look through the notebook globals for a list/dict of K splits whose i-th element of each split
    partitions range(n) exactly once -- that element is the test fold."""
    for name, obj in list(globals().items()):
        if name.startswith('_') or not isinstance(obj, (list, tuple, dict)):
            continue
        items = list(obj.values()) if isinstance(obj, dict) else list(obj)
        if not (2 <= len(items) <= 50):
            continue
        if all(isinstance(t, (list, tuple)) and 2 <= len(t) <= 4 for t in items):
            for pos in range(len(items[0])):
                try:
                    parts = [np.asarray(t[pos]).ravel() for t in items]
                    allidx = np.concatenate(parts)
                except Exception:
                    continue
                if allidx.dtype.kind in 'iu' and len(allidx) == n and np.array_equal(np.sort(allidx), np.arange(n)):
                    return f"{name}[k][{pos}]", parts
        if all(isinstance(t, dict) for t in items):
            for key in IDX_KEYS:
                if all(key in t for t in items):
                    parts = [np.asarray(t[key]).ravel() for t in items]
                    allidx = np.concatenate(parts)
                    if allidx.dtype.kind in 'iu' and len(allidx) == n and np.array_equal(np.sort(allidx), np.arange(n)):
                        return f"{name}[k]['{key}']", parts
    return None


if TEST_IDX is not None:
    TEST_IDX = [np.asarray(t).ravel() for t in TEST_IDX]
    _src = "TEST_IDX (config)"
else:
    _hit = _find_test_idx()
    _src, TEST_IDX = (_hit if _hit else ("not found", None))
print("preds container:", type(P).__name__, "| first keys:", list(P)[:4] if hasattr(P, '__iter__') else '')
print("test folds taken from:", _src)


# ---------------------------------- 2. assemble out-of-fold scores ----------------------------------
ID_KEYS = ('row_id', 'rowid', 'row', 'id', 'idx', 'index', 'pos', 'sample_id', 'participant_id')
FOLD_KEYS = ('fold', 'fold_id', 'cv_fold', 'test_fold', 'k')
YT_KEYS = ('y_true', 'y_test', 'y', 'target', 'label', 'outcome')


def _describe(obj, depth=0):
    pad = '  ' * depth
    if isinstance(obj, pd.DataFrame):
        print(f"{pad}DataFrame {obj.shape}; columns = {list(obj.columns)[:12]}")
        print(obj.head(3).to_string())
    elif isinstance(obj, dict):
        print(f"{pad}dict, {len(obj)} keys: {list(obj)[:6]}")
        if obj and depth < 2:
            _describe(next(iter(obj.values())), depth + 1)
    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{type(obj).__name__} of length {len(obj)}")
        if obj and depth < 2:
            _describe(obj[0], depth + 1)
    else:
        print(f"{pad}{type(obj).__name__}", getattr(obj, 'shape', ''), getattr(obj, 'dtype', ''))


def _scores_of(a):
    a = np.asarray(a)
    if a.dtype.kind not in 'fiub':
        raise ValueError(f"expected numeric scores but got {a.dtype} (e.g. {a.ravel()[:2]}) -- "
                         "the entry is probably a table/dict I do not recognise; see the structure printed above")
    a = a.astype(float)
    return a[:, 1] if a.ndim == 2 and a.shape[1] == 2 else a.ravel()


def _positions(ids, yt):
    """row-id column -> positions in Y. Tries: 0-based positions, 1-based positions, labels of Y.index,
    and 'rows are in Y order'; each hypothesis must reproduce y_true exactly."""
    m = len(ids) if ids is not None else (len(yt) if yt is not None else -1)
    ytv = None if yt is None else np.asarray(yt).astype(int)
    cands, report = [], []
    if ids is not None and np.asarray(ids).dtype.kind in 'iuf':
        ia = np.asarray(ids)
        if np.all(np.isfinite(ia)) and np.all(ia == np.floor(ia)):
            ia = ia.astype(int)
            cands.append(('0-based positions', ia))
            cands.append(('1-based positions', ia - 1))
            if not (isinstance(Y, pd.Series) and Y.index.equals(pd.RangeIndex(n))):
                try:
                    lab = np.asarray(Y.index.get_indexer(ia))
                    cands.append(('labels of Y.index', lab))
                except Exception:
                    pass
    if m == n:
        cands.append(('rows in Y order', np.arange(n)))
    for name, pos in cands:
        valid = pos.min() >= 0 and pos.max() < n and len(np.unique(pos)) == len(pos)
        agree = float(np.mean(ytv == y[pos])) if (valid and ytv is not None) else float('nan')
        report.append((name, valid, agree))
        if valid and (ytv is None or agree == 1.0):
            if name != '0-based positions':
                print(f"  (row_id interpreted as: {name})")
            return pos
    raise ValueError("cannot map row_id to positions in Y. Diagnostics -> "
                     f"row_id: min={None if ids is None else np.min(ids)}, max={None if ids is None else np.max(ids)}, "
                     f"unique={None if ids is None else len(np.unique(ids))}, rows={m}; n(Y)={n}; "
                     f"Y.index={type(Y.index).__name__}[{Y.index.min() if len(Y) else ''}..{Y.index.max() if len(Y) else ''}]; "
                     f"hypotheses (name, valid, share of y_true==Y): {report}")


def _parse_df(df):
    """table of predictions -> (row ids or None, scores, y_true or None, fold or None)"""
    cols = {str(c).lower(): c for c in df.columns}
    pick = lambda keys: next((cols[k] for k in keys if k in cols), None)
    idc, fc, yc = pick(ID_KEYS), pick(FOLD_KEYS), pick(YT_KEYS)
    used = {c for c in (idc, fc, yc) if c is not None}
    sc = pick(PRB_KEYS)
    if sc is not None and (sc in used or df[sc].nunique() <= 2):
        sc = None
    if sc is None:
        cand = [c for c in df.columns if c not in used and pd.api.types.is_float_dtype(df[c]) and df[c].nunique() > 2]
        named = [c for c in cand if re.search(r'prob|pred|score|proba', str(c).lower())]
        pool = named or cand
        if len(pool) != 1:
            raise ValueError(f"cannot pick the score column from {list(df.columns)} (candidates: {pool})")
        sc = pool[0]
    ids = df[idc].to_numpy() if idc is not None else (df.index.to_numpy() if df.index.dtype.kind in 'iu' else None)
    return (ids, df[sc].to_numpy(float),
            None if yc is None else df[yc].to_numpy(), None if fc is None else df[fc].to_numpy())


def _unpack(e):
    """one fold's entry -> (test positions or None, scores, y_true or None)"""
    if isinstance(e, pd.DataFrame):
        ids, s, yt, _ = _parse_df(e)
        return (None if ids is None else _positions(ids, yt)), s, yt
    if isinstance(e, dict):
        i = next((e[k] for k in IDX_KEYS if k in e), None)
        s = next((e[k] for k in PRB_KEYS if k in e), None)
        if s is None:
            raise ValueError(f"cannot find scores in a fold entry with keys {list(e)}")
        yt = next((e[k] for k in ('y_true', 'y_test', 'y') if k in e), None)
        return (None if i is None else np.asarray(i).ravel()), _scores_of(s), yt
    if isinstance(e, (tuple, list)) and len(e) >= 2 and not np.isscalar(e[0]):
        arrs = [np.asarray(a) for a in e]
        idx = [a.ravel() for a in arrs if a.ndim == 1 and a.dtype.kind in 'iu' and a.max() > 1]
        yt = [a.ravel() for a in arrs if a.ndim == 1 and a.dtype.kind in 'iub' and a.max() <= 1]
        sc = [a for a in arrs if a.dtype.kind == 'f' and (a.ndim == 2 or len(np.unique(a)) > 2)]
        if len(sc) != 1:
            raise ValueError("fold entry is a tuple/list but I cannot tell which element holds the scores")
        return (idx[0] if idx else None), _scores_of(sc[0]), (yt[0] if yt else None)
    return None, _scores_of(e), None


def _oof(x):
    """one arm -> (length-n OOF vector, fold vector or None)"""
    if isinstance(x, pd.DataFrame):                              # one table with all n rows of this arm
        ids, s, yt, fv = _parse_df(x)
        pos = _positions(ids, yt)
        v = np.full(n, np.nan)
        v[pos] = s
        fo = None
        if fv is not None:
            fo = np.full(n, -1)
            fo[pos] = np.asarray(fv).astype(int)
        return v, fo
    if isinstance(x, pd.Series):
        x = x.to_numpy()
    if isinstance(x, np.ndarray) and (x.shape[0] == n) and (x.ndim == 1 or (x.ndim == 2 and x.shape[1] == 2)):
        return _scores_of(x), None
    entries = list(x.values()) if isinstance(x, dict) else list(x)
    if isinstance(x, dict) and all(isinstance(k, (int, np.integer)) for k in x):
        entries = [x[k] for k in sorted(x)]
    v, fo = np.full(n, np.nan), np.full(n, -1)
    have_idx = True
    for k, e in enumerate(entries):
        idx, s, yt = _unpack(e)
        if idx is None:
            if TEST_IDX is None:
                raise ValueError("predictions do not carry row indices and no test folds were found: "
                                 "set TEST_IDX = [te for tr, va, te in <your splits>] in the CONFIG block")
            idx = TEST_IDX[k]
        if len(idx) != len(s):
            raise ValueError(f"fold {k}: {len(idx)} test rows but {len(s)} scores")
        if yt is not None and not np.array_equal(np.asarray(yt).astype(int), y[idx]):
            raise ValueError(f"fold {k}: y_true stored with the predictions does not match Y[test rows] -> "
                             "the test folds I am using are not the ones used for these predictions")
        v[idx] = s
        fo[idx] = k
    return v, fo


SC = {}                                           # classifier -> (y, fold, {arm: out-of-fold scores})
_have = {k[1] for k in P if isinstance(k, tuple) and len(k) == 2}
_gone = [c for c in CLASSIFIERS if c not in _have]
if _gone: print('classifiers not present in preds and skipped:', _gone)
CLASSIFIERS = [c for c in CLASSIFIERS if c in _have]
for CLF in CLASSIFIERS:
    print(f"\n==================== assembling out-of-fold scores: {CLF} ====================")
    # --- normalise `preds` into {arm: something _oof understands}
    if isinstance(P, pd.DataFrame):
        lc = {str(c).lower(): c for c in P.columns}
        armc = next((lc[k] for k in ('arm', 'representation', 'method', 'features', 'feature_set', 'name', 'model')
                     if k in lc), None)
        clfc = next((lc[k] for k in ('classifier', 'clf', 'estimator', 'algo', 'algorithm') if k in lc), None)
        if armc is not None:                                         # one long table: rows = arm x participant
            d = P
            if clfc is not None:
                if CLF is None and d[clfc].nunique() > 1:
                    raise ValueError(f"preds holds several classifiers {list(d[clfc].unique())}: set CLF = '...' in CONFIG")
                if CLF is not None:
                    d = d[d[clfc] == CLF]
            P_items = {a: g.drop(columns=[c for c in (armc, clfc) if c is not None]) for a, g in d.groupby(armc, sort=False)}
        else:                                                        # wide table: one column per arm
            P_items = {c: P[c] for c in P.columns}
    else:
        _split = lambda k: (k[0], k[1]) if isinstance(k, tuple) and len(k) == 2 else (k, None)
        _clfs = sorted({c for _, c in map(_split, P) if c is not None}, key=str)
        if _clfs:                                                    # keys are (arm, classifier)
            if CLF is not None:
                _use = CLF
            else:
                _lr = [c for c in _clfs if re.search(r'logis|^lr$|logreg', str(c).lower())]
                if _lr:
                    _use = _lr[0]
                elif len(_clfs) == 1:
                    _use = _clfs[0]
                else:
                    raise ValueError(f"preds holds several classifiers {_clfs}: set CLF = '...' in CONFIG")
            print("classifier:", _use, "(available:", _clfs, ")")
            P_items = {_split(k)[0]: v for k, v in P.items() if _split(k)[1] == _use}
        else:
            P_items = dict(P)

    def _assemble_tables(items):
        """Preferred path: every arm is a table with row_id, fold, y and a score. Participants are matched across arms
        by row_id, and y / fold are taken from the tables themselves, so Y in memory is not needed for alignment."""
        ref, out = None, {}
        for arm, df in items.items():
            ids, sc_, yt, fv = _parse_df(df)
            if ids is None or yt is None or fv is None:
                return None
            o = np.argsort(ids, kind='mergesort')
            ids, sc_ = np.asarray(ids)[o], np.asarray(sc_)[o]
            yt, fv = np.asarray(yt)[o].astype(int), np.asarray(fv)[o]
            if len(np.unique(ids)) != len(ids):
                raise ValueError(f"arm '{arm}': duplicated row_id")
            if np.isnan(sc_).any():
                raise ValueError(f"arm '{arm}': missing scores")
            if ref is None:
                ref = (ids, yt, fv)
            elif not (np.array_equal(ids, ref[0]) and np.array_equal(yt, ref[1]) and np.array_equal(fv, ref[2])):
                raise ValueError(f"arm '{arm}' covers different participants / labels / folds than the first arm")
            out[arm] = sc_
        _, fold_ = np.unique(ref[2], return_inverse=True)
        return ref[1], fold_, out


    scores, fold = {}, None
    try:
        _tab = _assemble_tables(P_items) if P_items and all(isinstance(x, pd.DataFrame) for x in P_items.values()) else None
        if _tab is not None:
            y, fold, scores = _tab
            n = len(y)
            print(f"using row_id / fold / y stored in preds (Y in memory: n={len(Y)}, events={int(np.asarray(Y).sum())}"
                  f" -> {'consistent' if len(Y) == n and int(np.asarray(Y).sum()) == int(y.sum()) else 'DIFFERENT, ignoring it'})")
        else:
            fold = np.full(n, -1)
            for arm, x in P_items.items():
                v, fo = _oof(x)
                if np.isnan(v).any():
                    raise ValueError(f"arm '{arm}': {int(np.isnan(v).sum())} participants have no out-of-fold prediction")
                scores[arm] = v
                if fo is not None and (fold < 0).all() and (fo >= 0).all():
                    fold = fo
    except Exception:
        print("\n---- structure of preds as I see it (send this if the error is unclear) ----")
        _describe(P)
        raise
    if (fold < 0).any():
        if TEST_IDX is None:
            raise ValueError("cannot determine test folds: set TEST_IDX in the CONFIG block")
        for k, i in enumerate(TEST_IDX):
            fold[i] = k
    assert (fold >= 0).all(), "some participants are in no test fold"
    print(f"{len(scores)} arms | n = {n} | events = {int(y.sum())} | folds = {len(np.unique(fold))} "
          f"(events per fold {np.bincount(fold, weights=y).astype(int).tolist()})")
    SC[CLF] = (np.asarray(y).copy(), np.asarray(fold).copy(), dict(scores))

_c0 = CLASSIFIERS[0]
y, fold = SC[_c0][0], SC[_c0][1]
for _c in CLASSIFIERS[1:]:
    assert np.array_equal(SC[_c][0], y) and np.array_equal(SC[_c][1], fold), f'{_c}: different labels / folds than {_c0}'
    _miss = set(SC[_c][2]) - set(SC[_c0][2])          # a secondary classifier may cover FEWER arms (tuned CatBoost: TCB_ARMS), never more
    assert not _miss, f'arms available for {_c} but not for the primary classifier: {sorted(_miss)}'
scores = SC[_c0][2]                               # arm keys are identical for every classifier
n = len(y)


# ----------------------------------------- 3. bootstrap core -----------------------------------------
def _prep(yf, s):
    order = np.argsort(-s, kind='mergesort')
    ss = s[order]
    starts = np.r_[0, np.flatnonzero(ss[1:] != ss[:-1]) + 1]      # tie-group starts
    return order, yf[order].astype(float), starts


def _auc_ap(W, prep):
    """AUROC / average precision for every row of W (bootstrap counts); tie-aware, matches sklearn."""
    order, ys, starts = prep
    Ws = W[:, order]
    gp = np.add.reduceat(Ws * ys, starts, axis=1)
    gn = np.add.reduceat(Ws * (1.0 - ys), starts, axis=1)
    Pw, Nw = gp.sum(1), gn.sum(1)
    cp, cn = np.cumsum(gp, 1), np.cumsum(gn, 1)
    auc = (gp * ((Nw[:, None] - cn) + 0.5 * gn)).sum(1) / (Pw * Nw)
    ap = ((gp / Pw[:, None]) * (cp / np.maximum(cp + cn, 1.0))).sum(1)
    return auc, ap


def bootstrap_arms(y, fold, scores, B=2000, seed=12345, chunk=200):
    names = list(scores)
    folds = np.unique(fold)
    nf = len(folds)
    rng = np.random.default_rng(seed)
    oa, op = np.zeros(len(names)), np.zeros(len(names))
    ba, bp = np.zeros((B, len(names))), np.zeros((B, len(names)))
    for f in folds:
        idx = np.flatnonzero(fold == f)
        m_f, yf = idx.size, y[idx]
        prep = [_prep(yf, scores[k][idx]) for k in names]
        for j, p in enumerate(prep):
            a, ap = _auc_ap(np.ones((1, m_f)), p)
            oa[j] += a[0] / nf
            op[j] += ap[0] / nf
        for b0 in range(0, B, chunk):
            m = min(chunk, B - b0)
            W = np.stack([np.bincount(rng.integers(0, m_f, m_f), minlength=m_f) for _ in range(m)]).astype(float)
            for j, p in enumerate(prep):                       # same W for every arm -> paired
                a, ap = _auc_ap(W, p)
                ba[b0:b0 + m, j] += a / nf
                bp[b0:b0 + m, j] += ap / nf
    return names, oa, op, ba, bp


def _holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj


def _bh(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 1.0
    for r, i in enumerate(np.argsort(p)[::-1]):
        run = min(run, p[i] * m / (m - r)); adj[i] = run
    return adj


def contrast_table(names, obs, boot, pairs):
    """first minus second; percentile 95% CI; p from bootstrap-SE z-test (used for Holm/BH, no B-limited floor);
    p_pct = percentile-based p as a cross-check."""
    ix = {k: i for i, k in enumerate(names)}
    cols = ['delta', 'lo', 'hi', 'se', 'p_z', 'p_pct', 'p_holm', 'p_bh']
    if not pairs:
        return pd.DataFrame(columns=cols, dtype=float)
    rows = []
    for a, b in pairs:
        d0 = obs[ix[a]] - obs[ix[b]]
        d = boot[:, ix[a]] - boot[:, ix[b]]
        lo, hi = np.percentile(d, [2.5, 97.5])
        se = d.std(ddof=1)
        Bn = len(d)
        rows.append(dict(comparison=f"{a}  vs  {b}", delta=d0, lo=lo, hi=hi, se=se,
                         p_z=2 * norm.sf(abs(d0) / se) if se > 0 else 1.0,
                         p_pct=min(1.0, 2 * min((1 + (d <= 0).sum()) / (Bn + 1), (1 + (d >= 0).sum()) / (Bn + 1)))))
    t = pd.DataFrame(rows).set_index('comparison')
    t['p_holm'], t['p_bh'] = _holm(t['p_z'].values), _bh(t['p_z'].values)
    return t


# --------------------------------- 4. map arms to roles, define families ---------------------------------
def _tok(k):
    """arm key -> tokens, dropping the k=50 tag and classifier tags (oao_50_onset -> ['oao', 'onset'])"""
    t = re.sub(r'(^|[^0-9])50(?![0-9])', r'\1 ', str(k).lower())
    return [w for w in re.split(r'[^a-z0-9]+', t) if w and w not in ('k', 'lr', 'logreg', 'logisticregression')]


ALIAS = dict(
    raw=['raw'], onset=['rawicd10onset', 'rawonset', 'onset', 'icd10onset', 'rawplusonset', 'rawplusicd10onset'],
    oao=['oao'], mdbt=['mdbt'], dceg=['dceg'],
    rand=['randomgraph', 'random', 'randgraph'],
    agesex=['ageandsexgraph', 'agesexgraph', 'ageandsex', 'agesex'],
    shuf=['shuffledhistorydceg', 'shuffledhistory', 'shuffled', 'shuf', 'shufdceg'],
    oao_on=['oaoonset', 'oaoicd10onset', 'oaoplusonset'], mdbt_on=['mdbtonset', 'mdbticd10onset', 'mdbtplusonset'],
    dceg_on=['dcegonset', 'dcegicd10onset', 'dcegplusonset'],
    presence=['rawpresenceonly', 'presenceonly', 'presence', 'rawpresence'], code_count=['rawcodecount', 'codecount', 'rawcount'],
    count_group=['rawcountsbycontentgroup', 'countsbycontentgroup', 'countsbygroup', 'countgroup', 'rawcountgroups', 'countgroups'],
    no_f41=['rawonsetnof41f43', 'nof41f43', 'nof41'], no_chF=['rawonsetnochapterf', 'nochapterf'],
    no_pc=['rawonsetnoprimarycaretype', 'noprimarycaretype', 'nopc'],
    oao500=['oaopool500timed', 'oaopool500', 'oao500'],
    wj500=['weightedjaccardpool500untimed', 'weightedjaccardpool500', 'wjpool500', 'wj500'],
    fknn=['featureknncontrol', 'featureknn', 'fknn', 'knncontrol', 'featknn'],
    shuf_sex=['shuffledhistorywithinsex', 'shuffledwithinsex', 'shufflewithinsex'],
    mlte=['mlte'], mlte_on=['mlteonset'], mltevec=['rawmltevec', 'mltevec'],
    oao_nof=[], mdbt_nof=[], dceg_nof=[], oao_on_nof=[], mdbt_on_nof=[], dceg_on_nof=[],   # always mapped explicitly via NAME
)


def _match(role, T):
    """looser token rules, used only when no exact alias matches"""
    S = set(T)
    has = lambda *w: any(x in S for x in w)
    sub = lambda w: any(w in t for t in T)
    on = has('onset', 'onsets')
    shuf = has('shuffled', 'shuffle', 'shuf')
    if role == 'raw': return T == ['raw']
    if role == 'onset': return on and S <= {'raw', 'icd10', 'icd', 'onset', 'onsets', 'ages', 'plus', 'age'}
    for gname in ('oao', 'mdbt', 'dceg'):
        if role == gname: return S == {gname}
        if role == gname + '_on': return gname in S and on and S <= {gname, 'onset', 'onsets', 'icd10', 'plus', 'raw'}
    if role == 'rand': return sub('random') and not on
    if role == 'agesex': return (('age' in S and 'sex' in S) or has('agesex', 'demographic', 'demo')) and not shuf and not on
    if role == 'shuf': return shuf and not has('sex', 'within') and not on
    if role == 'shuf_sex': return shuf and has('sex', 'within')
    if role == 'presence': return has('presence')
    if role == 'code_count': return has('code') and has('count', 'counts')
    if role == 'count_group': return has('count', 'counts') and has('group', 'groups', 'content')
    if role == 'no_f41': return sub('f41') or sub('f43')
    if role == 'no_chF': return ('f' in S and has('chapter', 'ch')) or sub('chapterf') or sub('chf')
    if role == 'no_pc': return has('primary', 'pc') or sub('primarycare')
    if role == 'oao500': return 'oao' in S and sub('500') and not has('untimed', 'wj', 'jaccard') and not on
    if role == 'wj500': return (sub('wj') or sub('jaccard') or has('untimed')) and sub('500')
    if role == 'fknn': return sub('knn')
    return False


R = {}
for role in ALIAS:
    if role in NAME:
        R[role] = NAME[role] if NAME[role] in scores else None
        if NAME[role] is not None and R[role] is None:
            print(f"  WARNING: {role} -> '{NAME[role]}' is not in preds")
        continue
    if NAME.get(role):
        R[role] = NAME[role]
        continue
    hits = [k for k in scores if ''.join(_tok(k)) in ALIAS[role]] or [k for k in scores if _match(role, _tok(k))]
    if len(hits) > 1:
        print(f"  note: {role} matched several arms {hits}; using {min(hits, key=lambda k: len(str(k)))}")
    R[role] = min(hits, key=lambda k: len(str(k))) if hits else None
print("\narm mapping (role -> key in preds):")
for role, k in R.items():
    print(f"  {role:12s} -> {k}")
print("arms in preds not used in any contrast:", [k for k in scores if k not in R.values()])
print("roles with no arm (their contrasts are skipped; map them in NAME):", [r for r, k in R.items() if k is None] or "none")
if not USE_ALL:
    _keep = set(R.values())
    for _c in SC:
        SC[_c] = (SC[_c][0], SC[_c][1], {k: v for k, v in SC[_c][2].items() if k in _keep})


def _pairs(spec, avail=None):
    """contrasts whose two arms exist (and, if `avail` is given, were run for this classifier)"""
    ok = lambda r: R[r] is not None and (avail is None or R[r] in avail)
    out = [(R[a], R[b]) for a, b in spec if ok(a) and ok(b)]
    skipped = [f'{a} vs {b}' for a, b in spec if not (ok(a) and ok(b))]
    print(f"  {len(out)}/{len(spec)} contrasts available" + (f"; SKIPPED: {skipped}" if skipped else ""))
    assert not (STRICT_FAMILIES and skipped and avail is None), f'pre-specified contrasts cannot be computed: {skipped}'
    return out


g, c, comb = ['oao', 'mdbt', 'dceg'], ['rand', 'agesex', 'shuf'], ['oao_on', 'mdbt_on', 'dceg_on']
MAIN = ([('onset', 'raw')] + [(x, 'raw') for x in g + c] + [(x, z) for z in c for x in g] + [('onset', x) for x in g]
        + [(x, 'onset') for x in comb] + list(zip(comb, g)) + [('oao', 'dceg'), ('oao', 'mdbt'), ('dceg', 'mdbt')]
        + [('oao_on', 'dceg_on'), ('oao_on', 'mdbt_on'), ('dceg_on', 'mdbt_on')])
SENS = [('wj500', 'oao500'), ('count_group', 'presence'), ('count_group', 'code_count'),
        ('no_f41', 'no_chF'), ('no_f41', 'no_pc'), ('fknn', 'shuf_sex')]
assert len(MAIN) == 31 and len(SENS) == 6
# MLTE family: added after the main analysis, so it is its own family (own Holm/BH) and leaves the 31 / 6 adjustments untouched
MLTE_FAM = [('mlte', 'raw'), ('mlte', 'onset'), ('mlte', 'rand'), ('mlte', 'agesex'), ('mlte', 'shuf'),
            ('mlte', 'oao'), ('mlte', 'mdbt'), ('mlte', 'dceg'),
            ('mlte_on', 'onset'), ('mlte_on', 'mlte'),
            ('mltevec', 'raw'), ('mltevec', 'mlte'), ('mltevec', 'onset')]       # mltevec vs mlte = embedding as columns vs as a graph
# F41/F43-removal family: added after the main analysis, so it is its own family (own Holm/BH).
# *_nof = graph built without F41/F43; *_on_nof = that graph + onset columns without F41/F43; no_f41 = raw + onset columns without F41/F43.
NOF_FAM = ([(f'{x}_nof', x) for x in g]                              # graph without F41/F43 minus the full graph
           + [(f'{x}_nof', r) for x in g for r in ('raw', 'rand')]   # does it still beat raw features / the random graph?
           + [(f'{x}_on_nof', f'{x}_on') for x in g]                 # combined arm without F41/F43 minus the full combined arm
           + [(f'{x}_on_nof', 'no_f41') for x in g]                  # does the graph add to onset columns (both without F41/F43)?
           + [(f'{x}_on_nof', f'{x}_nof') for x in g]                # do onset columns add to the graph (both without F41/F43)?
           + [('no_f41', 'raw')])                                    # gain of onset columns over raw features once F41/F43 are removed
assert len(NOF_FAM) == 19
# Logistic regression minus CatBoost, per representation (paired: same resamples); one family per list
LRCB_FAMILIES = {'main': ['raw', 'onset', 'oao', 'mdbt', 'dceg', 'rand', 'agesex', 'shuf', 'oao_on', 'mdbt_on', 'dceg_on'],
                 'mlte': ['mlte', 'mlte_on', 'mltevec'],
                 'nof': ['oao_nof', 'mdbt_nof', 'dceg_nof', 'oao_on_nof', 'mdbt_on_nof', 'dceg_on_nof']}

# --------------------------------------------- 5. run ---------------------------------------------
TUNED_NAME, LR_NAME, CB_NAME, CBT_NAME = 'LogisticRegressionTuned', 'LogisticRegression', 'CatBoost', 'CatBoostTuned'
_SFX = {TUNED_NAME: ''}                                 # PRIMARY (tuned LR) keeps the unsuffixed file names; the others get a suffix
sfx = lambda clf: _SFX.get(clf, '_' + re.sub(r'[^A-Za-z0-9]+', '', clf).lower())

# ONE bootstrap over (arm x classifier). Every column sees the same resampled participants, so contrasts within a
# classifier and all between-classifier contrasts are paired (per-arm results do not depend on which other arms are included).
joint = {f'{a}@{c}': SC[c][2][a] for c in CLASSIFIERS for a in SC[c][2]}
jn, joa, jop, jba, jbp = bootstrap_arms(y, fold, joint, B=B, seed=SEED)
_jix = {k: i for i, k in enumerate(jn)}

def _slice(clf):
    arms = list(SC[clf][2])                             # a classifier may cover fewer arms than the primary (tuned CatBoost)
    cols = [_jix[f'{a}@{clf}'] for a in arms]
    return arms, joa[cols], jop[cols], jba[:, cols], jbp[:, cols]

pd.set_option('display.width', 200)
q = lambda a: np.percentile(a, [2.5, 97.5], axis=0)
summ = lambda nm, t: print(f"{nm}: raw p<.05 (z): {(t.p_z < .05).sum()} | CI excludes 0: {((t.lo > 0) | (t.hi < 0)).sum()} | "
                           f"Holm<.05: {(t.p_holm < .05).sum()} | BH<.05: {(t.p_bh < .05).sum()} (of {len(t)})")
saved = []
for clf in CLASSIFIERS:
    names, oa, op, ba, bp = _slice(clf)
    print("\n" + "=" * 24 + f" {clf}" + (" (PRIMARY)" if clf == _c0 else "") + " " + "=" * 24)
    (la, ha), (lp, hp) = q(ba), q(bp)
    boot_arms = pd.DataFrame({'AUROC': oa, 'AUROC_lo': la, 'AUROC_hi': ha,
                              'AUPR': op, 'AUPR_lo': lp, 'AUPR_hi': hp}, index=names).round(4)
    print("Per-arm fold-mean metrics with patient-level bootstrap 95% CI "
          "(CHECK: point estimates should match results_summary_*.csv)")
    print(boot_arms.to_string())
    print("\nMain family (31):")
    boot_main_auroc = contrast_table(names, oa, ba, _pairs(MAIN, names))
    boot_main_aupr = contrast_table(names, op, bp, _pairs(MAIN, names))
    print("Sensitivity family (6; contrasts whose arms were not run are skipped):")
    boot_sens_auroc = contrast_table(names, oa, ba, _pairs(SENS, names))
    print(f"MLTE family ({len(MLTE_FAM)}; added after the main analysis, own Holm/BH):")
    boot_mlte_auroc = contrast_table(names, oa, ba, _pairs(MLTE_FAM, names))
    boot_mlte_aupr = contrast_table(names, op, bp, _pairs(MLTE_FAM, names))
    print(f"F41/F43-removal family ({len(NOF_FAM)}; added after the main analysis, own Holm/BH):")
    boot_nof_auroc = contrast_table(names, oa, ba, _pairs(NOF_FAM, names))
    boot_nof_aupr = contrast_table(names, op, bp, _pairs(NOF_FAM, names))
    print("\nMAIN AUROC contrasts (delta = first minus second):")
    print(boot_main_auroc.round(4).to_string())
    print("\nSENSITIVITY AUROC contrasts:")
    print(boot_sens_auroc.round(4).to_string())
    print("\nMLTE AUROC contrasts:")
    print(boot_mlte_auroc.round(4).to_string())
    print("\nF41/F43-removal AUROC contrasts:")
    print(boot_nof_auroc.round(4).to_string())
    for nm, t in [('main AUROC', boot_main_auroc), ('main AUPR', boot_main_aupr), ('sens AUROC', boot_sens_auroc),
                  ('MLTE AUROC', boot_mlte_auroc), ('MLTE AUPR', boot_mlte_aupr),
                  ('F41/F43-removal AUROC', boot_nof_auroc), ('F41/F43-removal AUPR', boot_nof_aupr)]:
        if len(t): summ(nm, t)
    for stem, t in [('arms', boot_arms), ('main_auroc', boot_main_auroc), ('main_aupr', boot_main_aupr),
                    ('sens_auroc', boot_sens_auroc), ('mlte_auroc', boot_mlte_auroc), ('mlte_aupr', boot_mlte_aupr),
                    ('nof_auroc', boot_nof_auroc), ('nof_aupr', boot_nof_aupr)]:
        fn = os.path.join(RESULTS_DIR, f'bootstrap_{stem}{sfx(clf)}.csv')
        t.to_csv(fn); saved.append(fn)

# ---------------- paired contrasts BETWEEN classifiers (first minus second, same arm, same resamples; Holm/BH within family)
def clf_minus(c1, c2):
    """one table per metric: for every arm run under BOTH classifiers, c1 minus c2, with Holm/BH within each arm family"""
    n1, oa1, op1, ba1, bp1 = _slice(c1)
    n2, oa2, op2, ba2, bp2 = _slice(c2)
    i1, i2 = {k: i for i, k in enumerate(n1)}, {k: i for i, k in enumerate(n2)}
    res = {}
    for metric, (o1, b1, o2, b2) in dict(auroc=(oa1, ba1, oa2, ba2), aupr=(op1, bp1, op2, bp2)).items():
        out = []
        for fam, roles in LRCB_FAMILIES.items():
            arms = [R[r] for r in roles if R.get(r) is not None and R[r] in i1 and R[r] in i2]
            rows = []
            for a in arms:
                d0 = o1[i1[a]] - o2[i2[a]]
                d = b1[:, i1[a]] - b2[:, i2[a]]
                lo, hi = np.percentile(d, [2.5, 97.5]); se = d.std(ddof=1)
                rows.append(dict(arm=a, family=fam, delta=d0, lo=lo, hi=hi, se=se,
                                 p_z=2 * norm.sf(abs(d0) / se) if se > 0 else 1.0,
                                 p_pct=min(1.0, 2 * min((1 + (d <= 0).sum()) / (len(d) + 1), (1 + (d >= 0).sum()) / (len(d) + 1)))))
            if rows:
                t = pd.DataFrame(rows)
                t['p_holm'], t['p_bh'] = _holm(t['p_z'].values), _bh(t['p_z'].values)
                out.append(t)
        res[metric] = pd.concat(out).set_index('arm') if out else pd.DataFrame()
    return res

CLF_CONTRASTS = [   # (first, second, file stem)
    (TUNED_NAME, LR_NAME, 'tuned_vs_lr'),            # tuned vs default L2 logistic regression  (regularisation effect)
    (TUNED_NAME, CB_NAME, 'tuned_vs_catboost'),      # primary vs default boosting
    (TUNED_NAME, CBT_NAME, 'tuned_vs_catboosttuned'),  # primary vs TUNED boosting  (the new comparison)
    (CBT_NAME, CB_NAME, 'catboosttuned_vs_catboost'),  # effect of tuning the booster
    (LR_NAME, CB_NAME, 'lr_vs_catboost'),            # legacy: default L2 LR vs default CatBoost
]
for c1, c2, stem in CLF_CONTRASTS:
    if c1 not in SC or c2 not in SC:
        print(f"\n(skipping {c1} minus {c2}: classifier not in preds)")
        continue
    print("\n" + "=" * 24 + f" {c1} minus {c2} " + "=" * 24)
    tabs = clf_minus(c1, c2)
    for metric, t in tabs.items():
        print(f"{metric.upper()} (Holm/BH within family):"); print(t.round(4).to_string())
        fn = os.path.join(RESULTS_DIR, f'bootstrap_{stem}_{metric}.csv'); t.to_csv(fn); saved.append(fn)

print(f"\nsaved to {RESULTS_DIR}:\n  " + "\n  ".join(saved))


## Repeated 10-fold cross-validation (reduced, resumable) — sensitivity analysis
Repeat 0 is the main run; this cell adds `RCV_EXTRA_REPEATS` new split seeds for the two logistic regressions (tuned = primary, default L2 = secondary) and the main arms only, then reports corrected resampled-t contrasts over all repeats x folds. It is a **sensitivity analysis**; the primary intervals come from the main partition (corrected-t over ten folds, and the patient-level bootstrap). Run it AFTER the main run (it reuses `results_df`, `iter_arms`, `evaluate_arm`).

In [ ]:
# =====================================================================================
#  REPEATED 10-FOLD CROSS-VALIDATION  (reduced, resumable, time-capped)  -- SENSITIVITY ANALYSIS, not the primary inference.
#  The primary interval is the corrected resampled t over the ten folds of the main run (plus the patient-level bootstrap);
#  the repeated design is reported as a sensitivity analysis of how much training-set variability the single partition hides.
#
#  Why: the corrected resampled-t interval ALLOWS for training-set variability but does not MEASURE it.  Repeating the
#  10-fold CV with new split seeds measures it directly (Bouckaert & Frank 2004: corrected resampled t over R x K folds).
#
#  How the run time is kept in check
#   * Repeat 0 IS the main run (same seed, same splits -- asserted below), so only RCV_EXTRA_REPEATS new repeats are run.
#   * Logistic regressions only (RCV_MODELS); neither CatBoost variant is repeated.
#   * Only the 11 main representations + 3 MLTE arms + two graph-free post hoc arms (counts, F41/F43 ablation) are run.
#   * HNSW recall checks and graph reports are switched off during the repeats.
#   * One tiny checkpoint per (repeat, fold): the cell is resumable, and RCV_MAX_MINUTES stops it cleanly after a fold.
#   * After the first computed fold the projected total time is printed -- interrupt there if it is too much and cut
#     RCV_EXTRA_REPEATS / the arm list.
#  Everything below (splits, graphs seeds, onset coding, model settings) is identical to the main run except the seeds.
# =====================================================================================
import pickle
from scipy import stats

# ------------------------- CONFIG -------------------------
# RCV_EXTRA_REPEATS (repeats added to the main run) and make_rcv_splits() are defined in the split-export cell after cell 4,
# so the exported split indices and this cell can never disagree.
assert 'make_rcv_splits' in globals() and 'RCV_EXTRA_REPEATS' in globals(), 'run the split-export cell (after cell 4) first'
RCV_MODELS = ['LogisticRegressionTuned', 'LogisticRegression']   # first = primary (tuned); default L2 kept as a secondary check.
# NB: checkpoints from the earlier default-L2-only run do not match this model list and are recomputed (checkpoint check includes `models`).
RCV_MAIN_ARMS = ['raw', 'raw_onset', 'oaopool500_50', 'mdbt_50', 'dceg_50', 'random_50', 'agesex_50', 'shufrow_dceg_50',
                 'oaopool500_50_onset', 'mdbt_50_onset', 'dceg_50_onset',
                 'mlte_50', 'mlte_50_onset', 'raw_mltevec']      # MLTE arms (embeddings trained per repeat and fold, checked by train_hash)
RCV_EXTRA_ARMS = ['raw_countgroups', 'raw_onset__no_F41_F43']      # no graph -> almost free; [] to drop
RCV_MAX_MINUTES = 360        # graceful stop (resumable) once this much wall-clock time has been spent in THIS call; None = no cap
RCV_ONLY_REPEATS = None      # e.g. [1, 2] to run only some repeats now; None = all of 1..RCV_EXTRA_REPEATS
NB_RATIO = None              # n_test / n_train in the corrected resampled t; None = taken from the splits
# ----------------------------------------------------------
RCV_ARMS = RCV_MAIN_ARMS + RCV_EXTRA_ARMS
assert set(RCV_ARMS) <= set(ALL_ARMS), sorted(set(RCV_ARMS) - set(ALL_ARMS))
RCV_CKPT_DIR = f'{preprocessed_dir}/ckpt_rcv_{BUILD_TAG}'
os.makedirs(RCV_CKPT_DIR, exist_ok=True)


def run_rcv_unit(rep, fold, rep_splits):
    """One (repeat, fold): build the arms one at a time, fit the models, keep metrics only."""
    global splits, PLAN, control_seed, hnsw_seed, CHECK_RECALL, REPORT_GRAPHS, CURRENT_REPEAT
    saved = (splits, PLAN, control_seed, hnsw_seed, CHECK_RECALL, REPORT_GRAPHS, CURRENT_REPEAT)
    try:
        CURRENT_REPEAT = rep
        splits = rep_splits                                            # make_fold_ctx reads this global
        PLAN = {a: list(RCV_MODELS) for a in RCV_ARMS}                 # evaluate_arm reads this global
        control_seed = saved[2] + 1000 * rep                           # random / shuffled controls and HNSW seeds differ per repeat
        hnsw_seed = control_seed
        CHECK_RECALL, REPORT_GRAPHS = False, False
        ctx = make_fold_ctx(fold)
        rows, fp = [], {}
        for arm, X_arm in iter_arms(ctx, set(RCV_ARMS)):
            evaluate_arm(arm, X_arm, ctx, rows, fp)
            del X_arm
        missing = [a for a in RCV_ARMS if (a, RCV_MODELS[0]) not in fp]
        assert not missing, f'arms not produced: {missing}'
        for r in rows:
            r['repeat'] = rep
        return rows
    finally:
        splits, PLAN, control_seed, hnsw_seed, CHECK_RECALL, REPORT_GRAPHS, CURRENT_REPEAT = saved


# ------------------------- run (resumable) -------------------------
reps_to_run = list(range(1, RCV_EXTRA_REPEATS + 1)) if RCV_ONLY_REPEATS is None else list(RCV_ONLY_REPEATS)
assert 0 not in reps_to_run, 'repeat 0 is the main run'
rcv_rows, t_start, n_computed, stopped = [], time.time(), 0, False
todo_units = [(r, f) for r in reps_to_run for f in range(num_folds)]
for rep in reps_to_run:
    rep_splits = make_rcv_splits(rep)
    for fold in range(num_folds):
        ck = f'{RCV_CKPT_DIR}/rep{rep}_fold{fold}.pkl'
        if os.path.exists(ck):
            d = pickle.load(open(ck, 'rb'))
            if d.get('build') == BUILD_TAG and d.get('arms') == RCV_ARMS and d.get('models') == RCV_MODELS:
                rcv_rows += d['rows']
                continue
        if RCV_MAX_MINUTES is not None and (time.time() - t_start) / 60 > RCV_MAX_MINUTES:
            stopped = True
            break
        t0 = time.time()
        rows = run_rcv_unit(rep, fold, rep_splits)
        pickle.dump({'build': BUILD_TAG, 'arms': RCV_ARMS, 'models': RCV_MODELS, 'rows': rows}, open(ck, 'wb'))
        rcv_rows += rows
        n_computed += 1
        left = sum(1 for r_, f_ in todo_units if not os.path.exists(f'{RCV_CKPT_DIR}/rep{r_}_fold{f_}.pkl'))
        per = (time.time() - t_start) / n_computed
        _log(f'RCV repeat {rep} fold {fold}: {time.time() - t0:.0f}s | avg {per / 60:.1f} min per fold | '
             f'{left} folds left -> about {left * per / 3600:.1f} h more')
    if stopped:
        break
if stopped:
    print(f'\nStopped after RCV_MAX_MINUTES={RCV_MAX_MINUTES}. Re-run this cell to resume (finished folds are loaded from checkpoints).')

# ------------------------- analysis (only repeats with all folds finished) -------------------------
if 'results_df' not in globals():
    results_df = pd.read_csv(os.path.join(RESULTS_DIR, f'results_per_fold_{BUILD_TAG}.csv'))
base = results_df[results_df['arm'].isin(RCV_ARMS) & results_df['model'].isin(RCV_MODELS)].copy()
base['repeat'] = 0
cols = ['arm', 'model', 'repeat', 'fold', 'auc', 'aupr']
rcv_df = pd.concat([base[cols], pd.DataFrame(rcv_rows)[cols] if rcv_rows else base[cols].iloc[:0]], ignore_index=True)
cnt = rcv_df.groupby('repeat')['fold'].nunique()
done_reps = [r for r in sorted(cnt.index) if cnt[r] == num_folds and rcv_df[rcv_df['repeat'] == r]['arm'].nunique() == len(RCV_ARMS)]
rcv_df = rcv_df[rcv_df['repeat'].isin(done_reps)]
rcv_df.to_csv(os.path.join(RESULTS_DIR, f'rcv_results_per_fold_{BUILD_TAG}.csv'), index=False)
R_DONE = len(done_reps)
print(f'\nrepeats complete: {done_reps}  ({R_DONE} x {num_folds}-fold = {R_DONE * num_folds} paired folds)')

def rcv_analysis(clf0, tag):
    """corrected-t analysis of the repeated CV for ONE classifier; `tag` is appended to the output file names"""
    print('\n' + '=' * 20 + f' repeated CV: {clf0} ' + '=' * 20)
    ratio = NB_RATIO if NB_RATIO is not None else float(np.mean([len(te) / len(tr) for tr, va, te in splits]))
    print(f'n_test / n_train used in the correction: {ratio:.4f}')

    def _wide(metric):
        w = rcv_df[rcv_df['model'] == clf0].pivot_table(index=['repeat', 'fold'], columns='arm', values=metric)
        assert not w.isna().any().any()
        return w

    def corrected_t(d, ratio, alpha=0.05):
        """Corrected resampled t (Nadeau & Bengio 2003; Bouckaert & Frank 2004 for repeated CV): var = (1/J + n_test/n_train) s^2, df = J-1."""
        d = np.asarray(d, float); J = len(d); m = d.mean(); s2 = d.var(ddof=1)
        se = np.sqrt((1.0 / J + ratio) * s2)
        tc = stats.t.ppf(1 - alpha / 2, J - 1)
        p = 2 * stats.t.sf(abs(m) / se, J - 1) if se > 0 else 1.0
        return m, m - tc * se, m + tc * se, p

    def _holm_r(p):
        p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
        for r_, i in enumerate(np.argsort(p)):
            run = max(run, (m - r_) * p[i]); adj[i] = min(1.0, run)
        return adj

    def _bh_r(p):
        p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 1.0
        for r_, i in enumerate(np.argsort(p)[::-1]):
            run = min(run, p[i] * m / (m - r_)); adj[i] = run
        return adj

    RR = dict(raw='raw', onset='raw_onset', oao='oaopool500_50', mdbt='mdbt_50', dceg='dceg_50', rand='random_50', agesex='agesex_50',
              shuf='shufrow_dceg_50', oao_on='oaopool500_50_onset', mdbt_on='mdbt_50_onset', dceg_on='dceg_50_onset',
              count_group='raw_countgroups', no_f41='raw_onset__no_F41_F43',
              mlte='mlte_50', mlte_on='mlte_50_onset', mltevec='raw_mltevec')
    g_, c_, comb_ = ['oao', 'mdbt', 'dceg'], ['rand', 'agesex', 'shuf'], ['oao_on', 'mdbt_on', 'dceg_on']
    MAIN_R = ([('onset', 'raw')] + [(x, 'raw') for x in g_ + c_] + [(x, z) for z in c_ for x in g_] + [('onset', x) for x in g_]
              + [(x, 'onset') for x in comb_] + list(zip(comb_, g_)) + [('oao', 'dceg'), ('oao', 'mdbt'), ('dceg', 'mdbt')]
              + [('oao_on', 'dceg_on'), ('oao_on', 'mdbt_on'), ('dceg_on', 'mdbt_on')])
    assert len(MAIN_R) == 31
    EXTRA_R = [('count_group', 'onset'), ('count_group', 'raw'), ('no_f41', 'raw'), ('no_f41', 'onset')]   # post hoc, own family

    # MLTE contrasts: own family (not part of the 31 pre-specified contrasts, Holm/BH applied within this family only)
    MLTE_R = [('mlte', 'raw'), ('mlte', 'rand'), ('mlte', 'agesex'), ('mlte', 'shuf'), ('mlte', 'mltevec'),
              ('mlte', 'oao'), ('mlte', 'mdbt'), ('mlte', 'dceg'),
              ('mlte_on', 'onset'), ('mlte_on', 'mlte'), ('mlte_on', 'oao_on'), ('mlte_on', 'mdbt_on'), ('mlte_on', 'dceg_on'),
              ('mltevec', 'raw')]

    def rcv_contrasts(metric, spec):
        w = _wide(metric)
        rep0 = w.loc[0]
        rows = []
        for a, b in spec:
            if RR[a] not in w.columns or RR[b] not in w.columns:
                continue
            d = (w[RR[a]] - w[RR[b]])
            m, lo, hi, p = corrected_t(d.values, ratio)
            _, lo0, hi0, _p0 = corrected_t((rep0[RR[a]] - rep0[RR[b]]).values, ratio)          # single partition (repeat 0) -> compare with the paper
            per_rep = d.groupby(level='repeat').mean()
            rows.append(dict(comparison=f'{RR[a]}  vs  {RR[b]}', delta=m, lo=lo, hi=hi, p=p, folds_ahead=int((d > 0).sum()),
                             n_folds=len(d), sd_of_repeat_means=per_rep.std(ddof=1), min_repeat_mean=per_rep.min(), max_repeat_mean=per_rep.max(),
                             rep0_lo=lo0, rep0_hi=hi0, width_ratio_vs_rep0=(hi - lo) / (hi0 - lo0)))
        t = pd.DataFrame(rows).set_index('comparison')
        if len(t):
            t['p_holm'], t['p_bh'] = _holm_r(t['p'].values), _bh_r(t['p'].values)
        return t

    arm_rows = []
    for metric in ('auc', 'aupr'):
        w = _wide(metric)
        per_rep = w.groupby(level='repeat').mean()
        for a in w.columns:
            arm_rows.append(dict(metric=metric, arm=a, mean=w[a].mean(), sd_over_folds=w[a].std(ddof=1),
                                 sd_of_repeat_means=per_rep[a].std(ddof=1), min_repeat_mean=per_rep[a].min(), max_repeat_mean=per_rep[a].max()))
    rcv_arms_t = pd.DataFrame(arm_rows).set_index(['metric', 'arm'])
    rcv_main_auroc, rcv_main_aupr = rcv_contrasts('auc', MAIN_R), rcv_contrasts('aupr', MAIN_R)
    rcv_extra_auroc, rcv_extra_aupr = rcv_contrasts('auc', EXTRA_R), rcv_contrasts('aupr', EXTRA_R)
    rcv_mlte_auroc, rcv_mlte_aupr = rcv_contrasts('auc', MLTE_R), rcv_contrasts('aupr', MLTE_R)

    pd.set_option('display.width', 250)
    print('\nPer-arm means over all repeats x folds, and how much the 10-fold mean moves between repeats:')
    print(rcv_arms_t.round(4).to_string())
    print(f'\nMAIN AUROC contrasts, corrected resampled t over {R_DONE} x {num_folds} folds (delta = first minus second):')
    print(rcv_main_auroc.round(4).to_string())
    print('\nPOST HOC AUROC contrasts (own family):')
    print(rcv_extra_auroc.round(4).to_string())
    print('\nMLTE AUROC contrasts (own family):')
    print(rcv_mlte_auroc.round(4).to_string())
    print('\nCHECK: rep0_lo / rep0_hi are the single-partition corrected-t intervals; they should reproduce the manuscript table '
          '(if not, set NB_RATIO to the ratio used there). width_ratio_vs_rep0 < 1 means repeated CV gave a narrower interval.')
    for stem, t in [('arms', rcv_arms_t), ('main_auroc', rcv_main_auroc), ('main_aupr', rcv_main_aupr),
                    ('extra_auroc', rcv_extra_auroc), ('extra_aupr', rcv_extra_aupr),
                    ('mlte_auroc', rcv_mlte_auroc), ('mlte_aupr', rcv_mlte_aupr)]:
        t.to_csv(os.path.join(RESULTS_DIR, f'rcv_{stem}{tag}.csv'))
    print(f'\nsaved to {RESULTS_DIR}: rcv_results_per_fold_*.csv, rcv_arms{tag}.csv, rcv_main_auroc{tag}.csv, rcv_main_aupr{tag}.csv, rcv_extra_auroc{tag}.csv, rcv_extra_aupr{tag}.csv, rcv_mlte_auroc{tag}.csv, rcv_mlte_aupr{tag}.csv')


if R_DONE >= 2:
    for _i, _clf in enumerate(RCV_MODELS):
        rcv_analysis(_clf, '' if _i == 0 else '_' + _clf.lower())     # primary (tuned) keeps the plain names
else:
    print('Fewer than 2 complete repeats: nothing to analyse yet.')
